# A real run, start to finish: how P-TEFb phosphorylates the RNA polymerase II CTD

This page follows one complete Octomod run on real data, from the question a
scientist brought to the model it arrived at. **Nothing here needs to be run.**
Every prompt, reply, check, number and figure on this page was read from the run's
single result file, `ctd_blind_local_v3_opus55_20261004_022720.h5` (the *record*); only the explanatory prose was
written by hand. The last section, [Explore the record yourself](explore_record.ipynb),
shows how to open the same file and find all of this directly.

| | |
|---|---|
| **The question** | Build a model of this CTD phosphorylation time course that explains the measured distributions, and say what it implies about how P-TEFb modifies the substrate. Report explicitly what these data can and cannot establish about the mechanism. In particular, test whether the modification of a repeat changes the rate at which its immediate neighbours in the array are modified, against the alternatives that the rate depends only on how many repeats are already modified (cooperativity between repeats) or on nothing at all, and say which of these the data can and cannot separate. Also test the earlier conclusion, drawn by eye, that the kinase acts distributively rather than processively. |
| **The data** | RNA polymerase II CTD (8 heptarepeats, in vitro), phosphorylated by P-TEFb (CDK9/cyclin T1); mass-spectrometry time course: 9 phosphorylation states × 4 time points, no replicates |
| **Agents** | 9 roles, 19 calls to Claude (`claude-opus-5-5`), $25.40 |
| **Decisions** | 15 answers at gates: 5 given at the card, 10 applied from standing answers the owner wrote down |
| **Time** | 21.9 h over 17 sessions: agents 2.8 h, fitting and other computation 2.0 h, waiting at gates 8.7 h |
| **Models fitted** | round 0: 5 candidate mechanisms; round 1: the round-0 winner and 1 extension of it; 100 optimiser starts each |
| **Outcome** | `completed`: **product_load_inhibition** (M5_neighbour_and_count + product_load_inhibition) selected decisively, Akaike weight 0.961 |


## 1 · The goal, from the user's side

The user is a biochemist with a small in-vitro dataset: the kinase **P-TEFb** was
mixed with a CTD construct of **8 repeats** and ATP, and mass spectrometry counted
how many repeats carried a phosphate after 0, 1, 2 and 16 hours. They want to know
*how* P-TEFb modifies the CTD, and in particular whether modifying one repeat
changes the rate at which its **neighbours** are modified, or whether the rate
depends only on **how many** repeats are already modified, or on **nothing at all**
— and they want to be told plainly what these data cannot decide.

The problem is a folder of plain files — a bench note, the data file and a short
intent (here `testcase/ctd_blind_local_v3`, a local copy of the `ctd_blind` benchmark, whose
published answer the agents never see). The front end (Claude Code in this
repository) started the engine with live agents and relayed every decision:

```bash
./run.sh run --testcase testcase/ctd_blind_local_v3 --backend llm --run-id ctd_blind_local_v3_opus55_20261004_022720 --gate-mode file
```

The run snapshotted these inputs into its record before the first module, so the
record holds exactly what the agents were given:

| file | what it is |
|---|---|
| blocklist.txt | words the run must not act on (a research blocklist) |
| data/ctd_ms_timecourse.csv | the measurements |
| description.md | the bench note, in the user’s words |
| intent.yaml | the frozen intent: question and success criteria |
| testcase.yaml | run settings (end time, no holdout, knowledge module off for this benchmark) |

No measurement is held back as a test: there is one condition and four time
points, and a held-out time point would test its own free scale factor rather
than the mechanism, as `testcase.yaml` explains. The candidates are therefore
compared on an information criterion alone.


::::{dropdown} The bench note (`description.md`), as the user wrote it
> # CTD phosphorylation by P-TEFb — in vitro time course (bench note)
>
> ## The system
>
> RNA polymerase II carries a long, flexible tail on its largest subunit called the
> C-terminal domain, or CTD. The CTD is not a folded domain: it is a tandem array of
> short repeats, each seven amino acids long, with the sequence Tyr–Ser–Pro–Thr–Ser–Pro–Ser.
> In human Pol II there are 52 of these heptarepeats. During transcription the CTD is
> progressively decorated with phosphate groups, and the resulting pattern of marks is
> what recruits and releases the RNA-processing and elongation factors that act on the
> polymerase. Which marks are present, and where, is therefore functionally important.
>
> The kinase we are studying here is \*\*P-TEFb\*\* (the CDK9–cyclin T1 complex), which acts
> during transcriptional elongation.
>
> ## The experiment
>
> This is an \*\*in vitro kinase assay\*\*, not a cellular experiment. There is no
> transcription, no DNA, no polymerase — only a purified CTD substrate, purified kinase,
> and ATP in buffer.
>
> - \*\*Substrate\*\*: a recombinant CTD construct containing \*\*8 heptarepeats\*\*, not the full
>   52. Present at \*\*100 µM\*\*.
> - \*\*Kinase\*\*: P-TEFb, catalytic. It is not consumed and its concentration does not change
>   over the experiment.
> - \*\*ATP\*\*: \*\*3 mM\*\* at the start. Each phosphorylation consumes one ATP.
> - \*\*Sampling\*\*: four time points — \*\*0, 1 h, 2 h and 16 h\*\* (0, 60, 120 and 960 minutes).
> - \*\*Replication\*\*: one measurement per species per time point. There are \*\*no replicate
>   measurements\*\* in this dataset.
>
> ## What the readout is, and what it cannot see
>
> The readout is \*\*mass spectrometry\*\* on the intact CTD construct. MS separates the
> substrate molecules by mass, and since each phosphate adds a fixed mass, it resolves
> \*\*how many of the eight repeats carry a phosphate\*\* — a number between 0 and 8.
>
> \*\*It does not resolve which repeats those are.\*\* A molecule phosphorylated on repeats 1
> and 2 and a molecule phosphorylated on repeats 1 and 7 are the same mass and appear in
> the same peak. Everything downstream has to live with that: the data are nine numbers per
> time point, one for each phosphorylation count, and the positional arrangement behind
> those numbers is not observed.
>
> A second point about the chemistry, established previously for this kinase and these
> conditions: within a single heptarepeat, phosphorylation happens on \*\*either Ser2 or
> Ser5, but not both at once\*\*. So for the purposes of counting, each repeat is either
> modified or unmodified — one bit per repeat — and the distinction between the two serine
> positions does not need to be tracked.
>
> ## The data file, and a caveat you must handle
>
> `data/ctd_ms_timecourse.csv` has one row per time point and nine columns, `phos0_rel`
> through `phos8_rel`: the relative MS signal for CTD molecules carrying 0, 1, … 8
> phosphates.
>
> \*\*These are relative signal intensities, not concentrations, and they are not comparable
> between time points as they stand.\*\* Mass spectrometry reports intensities on an
> arbitrary scale that drifts between injections. You can see this directly in the file:
> the nine numbers sum to 1.00 at t = 0, to 4.07 at 1 h, to 3.79 at 2 h and to 0.85 at
> 16 h, even though the total amount of substrate in the tube is conserved throughout.
> Any model of this data needs a way to map predicted amounts onto measured intensities,
> and that mapping is not the same at every time point.
>
> The measurement error is believed to be additive and normally distributed, with a single
> standard deviation for the whole experiment — the same for every peak — that scales from one
> time point to the next with that spectrum's overall intensity, the same factor that maps
> amounts onto intensities at that time point.
>
> ## What is already believed, and what is not settled
>
> Earlier work on this kinase, using this kind of hyperphosphorylation assay in human and
> in \*Drosophila\*, concluded that P-TEFb works \*\*distributively rather than processively\*\*:
> that it modifies a repeat, lets go, and rebinds — rather than holding on and running
> along the substrate. That conclusion came from looking at the shape of MS distributions
> by eye, and was never checked against a quantitative model.
>
> Beyond that, the mechanism is genuinely open. We do not know from these data what governs
> the rate at which any individual repeat gets modified, and that is what we would like a
> model to tell us. One specific possibility was put to us and we want it tested rather than
> assumed either way: that the modification of one repeat changes the rate at which its
> immediate neighbours in the array are modified, as opposed to the rate depending only on
> how many repeats of the molecule are already modified (cooperativity between repeats,
> which was also raised as a possibility to look at), or on nothing at all.
>
> ## What can be computed here
>
> Any model that simulates is acceptable, whether it tracks only how many repeats are
> modified or which repeats are modified (with eight repeats, 256 configurations). Models
> of a few hundred states and a thousand reactions simulate in seconds on the machine this
> runs on, so size alone is not a reason to prefer one description over another; choose
> the description the question needs. If a possibility can only be represented by tracking
> which repeats are modified, we would rather see that model fitted than the question
> settled by argument from the counts alone. Such a model can be written out in full, every
> state and every reaction listed one by one, and we would rather it were written out than
> avoided. The simulator reserves the single letters x, k, p, y, w, h and t, so no species
> or parameter may be named with one of them.
>
> ## The question
>
> Build a model of this time course that explains the measured distributions, and tell us
> what it implies about how P-TEFb modifies the CTD. Be explicit about what these data can
> and cannot establish — with eight repeats, four time points and no positional
> information, we expect there to be things the data simply cannot decide, and we would
> rather know which those are than have them glossed over. We would also like the earlier
> conclusion, drawn by eye, that the kinase acts distributively rather than processively,
> to be tested against a model and not assumed.
::::

### What counts as success

The intent is **frozen** before anything is built: every later verdict is scored
against it, and the agents may not change it.

- **Inference target:** `mechanism_discrimination` — pick between mechanisms, not predict a new condition
- **Required predictions:**
  - the mechanism of phosphorylation that best explains the measured distributions, identified and defended
  - a statement of which mechanistic questions these data cannot answer
- **Success criteria:**
  - the selected mechanism is reported with its information criterion, its Akaike weight and the indistinguishable set, honestly, whatever they turn out to be
  - the observation model is stated explicitly, including how predicted amounts are mapped onto relative MS intensities at each time point
  - any mechanistic question the data cannot resolve is named as unresolved rather than decided
- **Out of scope:**
  - transcription, DNA and the polymerase itself — this is a purified in vitro assay
  - the distinction between Ser2 and Ser5 within a repeat (established as mutually exclusive under these conditions; each repeat is one bit)

### The data

![The measured spectra](figures/data.svg)

Nine relative intensities per time point. Two things make this hard: the totals
drift between injections (1.00, 4.07, 3.79, 0.85) although the amount of CTD is
constant, so every time point needs its own scale; and the instrument sees *how
many* repeats are modified, never *which* ones.


## 2 · The run at a glance

The engine runs a fixed sequence of modules. Agents draft (the model, the reading
of the data, the observation model, the interpretation, and here a proposal for
what the model space lacked); deterministic checks test each draft; independent
checker agents audit them; and the run **stops at a gate** whenever the next step
costs compute or makes a scientific claim.

![Timeline of the run](figures/timeline.svg)

This run took 21.9 hours of wall time in
17 sessions: each gate ends a session, and the next one starts
when the answer arrives. The grey bands are those waits,
8.7 hours in all. The agents (orange) took
2.8 hours; fitting and the other computation took
2.0 hours.

**Every agent call** (click a role to read its full prompt and reply):

| agent | round | job | calls | prompt chars | output tokens | cost |
|---|---|---|---|---|---|---|
| [`construction`](prompts/construction.md) | 0 | builds the base model (S1) | 1 | 55,857 | 59,132 | $1.36 |
| [`expansion`](prompts/expansion.md) | 0 | turns the open questions into candidate models (S1) | 4 | 184,209 | 360,581 | $9.75 |
| [`construction_verification`](prompts/construction_verification.md) | 0 | independent check of the model and its candidates (S1) | 2 | 154,182 | 68,009 | $2.80 |
| [`data_extraction`](prompts/data_extraction.md) | 0 | reads the data file into measurements (S2) | 2 | 24,341 | 13,831 | $0.45 |
| [`observation`](prompts/observation.md) | 0 | maps model states onto what the instrument measures (S3) | 2 | 70,915 | 20,597 | $0.89 |
| [`condition`](prompts/condition.md) | 0 | encodes the experimental condition (S3) | 2 | 47,569 | 4,540 | $0.42 |
| [`assembly_verification`](prompts/assembly_verification.md) | 0 | independent check of the assembled estimation problem (S3) | 2 | 49,984 | 28,632 | $0.96 |
| [`interpretation`](prompts/interpretation.md) | 0 | reads the fits and answers the question (S7, round 0) | 1 | 340,245 | 46,470 | $2.24 |
| [`revision`](prompts/revision.md) | 0 | reads what the selected model still misses and proposes new mechanisms | 2 | 272,575 | 149,804 | $4.92 |
| [`interpretation`](prompts/interpretation_round_1.md) | 1 | reads the fits and answers the question (S7, round 1) | 1 | 203,192 | 42,139 | $1.61 |

**Every answer at a gate.** The first 5 were given at the
card. For the rest, the owner wrote down *standing answers* on 4 October —
approve compiling and the default estimation budget, expand once when the
interpreter recommends revising and the expansion's diagnosis names a missing
mechanism, and stop after that one round — and the front end applied them at
each card, naming the standing answer in the note. Every answer went through
the front end in file-gate mode and is recorded as a human decision (the
verdicts read "signed by: human (via the front end)"); none came
from a policy profile (the run's policy was `strict`).

| round | gate | what the card asked | answer | how | when |
|---|---|---|---|---|---|
| 0 | freeze the intent | Freeze this intent? Everything downstream is scored against it. | yes | at the card | 0.0 h (00:30 UTC) |
| 0 | model scope | Discriminate between these 5 candidate mechanisms? (each will be compiled and fitted) | yes — note: `1: M3 and M5 as built (the 74 exact gap classes) are the neighbour test; no base redraft` | at the card | 9.1 h (09:34 UTC) |
| 0 | assembly failed | assembly: a blocking check failed and no agent can be asked. Grant more attempts, send your own words, apply a declared fallback, or stop? | no | at the card | 9.7 h (10:11 UTC) |
| 0 | model scope | Discriminate between these 5 candidate mechanisms? (each will be compiled and fitted) | yes — note: `redraft: In M3_neighbour and M5_neighbour_and_count, the 74 gap-class states (Gf8, G07 ... G00) are already declared as species in the append block; remove them` | at the card | 10.3 h (10:48 UTC) |
| 0 | model scope | Discriminate between these 5 candidate mechanisms? (each will be compiled and fitted) | yes — note: `1: M3 and M5 as built (the 74 exact gap classes) are the neighbour test; no base redraft` | at the card | 10.7 h (11:11 UTC) |
| 0 | compile | Approve AMICI compile + cost probe for 5 candidates? | yes — note: `follows the owner's standing answers of 2026-10-04 (assembly_cost_boundary: y)` | standing answer | 11.9 h (12:23 UTC) |
| 0 | estimation budget | Run the full estimation with 100 starts x 5 candidates (projected ~898s total)? | yes — note: `follows the owner's standing answers of 2026-10-04 (budget: y, default starts, forward sensitivities)` | standing answer | 12.0 h (12:24 UTC) |
| 0 | estimation budget | Run the full estimation for candidate 'M3_neighbour' with 100 starts (projected ~137s wall)? Its probe differs from the approved candidate's — see the first lines. | yes — note: `follows the owner's standing answers of 2026-10-04 (budget: y, default starts, forward sensitivities)` | standing answer | 12.0 h (12:29 UTC) |
| 0 | estimation budget | Run the full estimation for candidate 'M5_neighbour_and_count' with 100 starts (projected ~179s wall)? Its probe differs from the approved candidate's — see the first lines. | yes — note: `follows the owner's standing answers of 2026-10-04 (budget: y, default starts, forward sensitivities)` | standing answer | 12.1 h (12:35 UTC) |
| 0 | verdict | Sign off this discrimination verdict? (it is a scientific claim) | yes — note: `expand: yes` | standing answer | 15.2 h (15:38 UTC) |
| 0 | buy an expansion round | Buy this expansion and run another selection round? | yes — note: `follows the owner's standing answers of 2026-10-04 (revision: y when the diagnosis names a missing mechanism; one expansion round)` | standing answer | 17.5 h (17:58 UTC) |
| 1 | model scope | Is this the right model scope to spend compute on? | yes — note: `tier: all` | standing answer | 17.5 h (17:59 UTC) |
| 1 | compile | Approve AMICI compile + cost probe for 2 candidates? | yes — note: `follows the owner's standing answers of 2026-10-04 (assembly_cost_boundary: y). For the record: processive_with_coupling was bought at the expansion gate and named ag` | standing answer | 18.7 h (19:11 UTC) |
| 1 | estimation budget | Run the full estimation with 100 starts (projected ~205s wall)? | yes — note: `follows the owner's standing answers of 2026-10-04 (budget: y, default starts, forward sensitivities)` | standing answer | 18.8 h (19:13 UTC) |
| 1 | verdict | Sign off this discrimination verdict? (it is a scientific claim) | yes — note: `expand: no` | standing answer | 21.8 h (22:13 UTC) |


## 3 · Round 0, step by step

### 3.1 · S0 — freezing the intent

The first stop, before any agent is called: the person confirms the question and
the success criteria, which are then frozen for the rest of the run.

:::{admonition} A gate · freeze the intent (round 0)
:class: important
**The card asked:** Freeze this intent? Everything downstream is scored against it.

**The answer:** **yes**, given at the card.
:::

### 3.2 · S1 — building the base model (`construction` agent)

The construction agent receives the bench note, the intent and two domain packs
(general dynamics, measurement technology) — [the full prompt](prompts/construction.md),
55,857 characters. It has no model to start
from: it must derive one from the note.

Its central decision was how to represent the substrate. Eight repeats, each
modified or not, give 256 configurations, observed only through their 9 counts.
The agent's base tracks only the **count** of modified repeats per molecule
(`CTD_0` … `CTD_8`). As long as every unmodified repeat of a molecule is modified
at the same rate, that lumping is exact. A **neighbour effect** is the one
hypothesis that breaks it, so for that candidate the agent wrote down an exact
rule to *generate* the positional model instead of writing it by hand.

In the base, each free repeat is modified at a rate
`kmod · PTEFb · ATP/(K_ATP + ATP)`, with switches for the hypotheses: `cp`
multiplies the rate by `cp` per mark already on the molecule (count
cooperativity), `q_proc` lets the kinase continue to another repeat before
releasing (processivity), and in the positional model `nb` multiplies it per
modified immediate neighbour. With every switch at its neutral value, all repeats
are modified independently at one rate — the "nothing at all" hypothesis.

| parameter | nominal | unit | role | why (the agent’s words) |
|---|---|---|---|---|
| `kmod` | 0.01 | 1/min per repeat per relative P-TEFb unit | estimated | order-of-magnitude default: a per-repeat rate of 0.01/min half-modifies repeats by ~70 min, inside the 60-960 min sampling window |
| `cp` | 1.0 | dimensionless (fold change of per-repeat rate per mark already on the molecule) | fixed / switch | switch for count cooperativity; 1 = off (neutral value) |
| `q_proc` | 0.0 | dimensionless probability of continuing to another repeat before release | fixed / switch | switch for processivity; 0 = off (strictly distributive, neutral value) |
| `K_ATP` | 50.0 | uM | fixed / switch | general knowledge: protein-kinase ATP Michaelis constants typically lie in the 10-100 uM range; not measured for this assay |

**The agent's decisions, each with its rationale** (click to expand):


::::{dropdown} The base tracks how many of the 8 repeats each molecule carries (CTD_0..CTD_8), not which. It is the exact count lumping of the 256-configuration positional …
> For every rate rule this base can switch on (independent repeats, count cooperativity cp^n, processive continuation that picks among the remaining unmodified repeats), all unmodified repeats of a molecule with n marks share one rate, so the 256 configurations are strongly lumpable by n and these 9 states reproduce the positional model's count distribution exactly. The 'nothing at all', 'cooperativity' and 'distributive vs processive' hypotheses therefore lose nothing by being counted. A nearest-neighbour effect is the one hypothesis that breaks the lumping, so it cannot be a switch on this base. Two previous replies to this brief failed the contract outright; a 1024-reaction hand enumeration, together with its 256-entry glossary and network, is not something this reply can carry reliably or check without execution. The next decision gives the exact rule so the positional model can be generated mechanically. A generator check comes free: at nb = 1 it must reproduce this base trajectory for trajectory.

**Alternatives considered.** Full 256-configuration model written out by hand as the base (the user's stated preference; not deliverable reliably here). Mirror-symmetry reduction to 136 orbits (exact, but still ~500 reactions with multiplicities and harder to audit). Pair or domain-count approximations (n, number of marked runs) were rejected because they are not exact, and the lab note asks that the question not be settled from counts alone.

*confidence: medium · evidence tier: default*
::::

::::{dropdown} Exact construction rule for the positional (neighbour) candidate, to be generated and not hand-written. Species C_b1b2b3b4b5b6b7b8 (b_i = 1 if repeat i, …
> This is the minimal positional extension that separates 'a mark changes its neighbours' rate' (nb) from 'the rate depends on how many marks the molecule carries' (cp) and from neither (nb = cp = 1). It is 8\*2^7 = 1024 reactions. Summing C over the C(8,n) configurations with n marks gives CTD_n, so readouts map onto it exactly as onto this base. nb \< 1 (neighbour inhibition) jams molecules near 3 to 4 non-adjacent marks before slow filling, and that stall at intermediate counts between 2 h and 16 h is something cp alone cannot shape. nb > 1 gives nucleation-and-growth (bimodal 0-versus-high counts at 1 to 2 h), which cp > 1 and processivity also give. That is the likely indistinguishable set.

**Alternatives considered.** A directional neighbour effect (separate factors for N- and C-side neighbours) was not chosen: counts carry no positional information to separate the two directions, so it would add a flat direction. Pairwise or longer-range interactions were not chosen either: they are not the hypothesis named.

*confidence: medium · evidence tier: to_be_estimated*
::::

::::{dropdown} Distributive single-repeat steps J0..J7. Each unmodified repeat is modified at kmod\*PTEFb\*ATP/(K_ATP + ATP), mass action in the substrate at constant enzyme, …
> The assay is purified kinase, CTD and ATP with no phosphatase, so removal of phosphate is not modelled. The kinase is catalytic and constant, so its level enters as a fixed multiplier. Mass action in CTD assumes the kinase is far from saturated by substrate. Neither the enzyme concentration nor a CTD Km is given, so saturation cannot be decided from the note. If all unmodified repeats bind alike, saturation would only reparameterise time by a factor common to every reaction (the same sequence of distributions, a different clock). With three post-zero time points that clock change is largely absorbed by kmod and cp, so it is named unresolved rather than built in. With cp = 1 and q_proc = 0 the count distribution is binomial(8, 1 - exp(-r\*t)): the independent-repeat ('nothing at all') hypothesis.

**Alternatives considered.** Michaelis-Menten in CTD with competition among all substrate forms (left out: time warp only, not identifiable here). Reversible steps (no phosphatase present).

*confidence: medium · evidence tier: default*
::::

::::{dropdown} Count-cooperativity switch cp: the per-repeat rate of a molecule carrying n marks is multiplied by cp^n. cp = 1 (base) turns it off; cp > 1 is positive and cp …
> One parameter expresses 'the rate depends only on how many repeats are already modified' as a constant fold change per existing mark, the additive free-energy form. It is the most parsimonious count-dependent law the data can constrain with 9 peaks at 3 informative time points. A free factor per n (7 parameters) would not be identifiable. Bounds 0.2 to 5 are 1.4 decades on cp but compound to ~10 decades on the n = 7 step. Wider bounds put the solver in stiff corners without adding plausible behaviour.

**Alternatives considered.** A free rate per count n (unidentifiable). A saturating dependence on n (an extra shape parameter the data cannot see).

*confidence: medium · evidence tier: to_be_estimated*
::::

::::{dropdown} Processivity switch q_proc on reactions Jp\<n>_\<j>. A binding event on a molecule with n marks starts at the same encounter rate as a distributive step. After …
> Distributive versus processive is about how many modifications one binding event makes. The encounter-level jump n -> n+j with geometric run length is the standard count-level representation, and its weights sum to 1, so q_proc changes what an encounter does, not how often encounters happen. Strong processivity predicts early bimodality (CTD_0 plus high counts, depleted intermediates at 1 h). Its difference from cp > 1 is memory: under processivity a released molecule returns to the slow distributive pace, so long-lived intermediates persist to the 16 h point. Sliding processivity (running along to the next repeat, stopped by an already-modified repeat) depends on positions and only exists in the positional model. It is not exactly lumpable and is likely aliased with nb > 1.

**Alternatives considered.** An explicit enzyme-substrate complex per state (adds unknown enzyme concentration and koff, which the data cannot see; the encounter is far shorter than the 60 min sampling). Sliding processivity (positional only, see previous decisions).

*confidence: medium · evidence tier: to_be_estimated*
::::

::::{dropdown} ATP is an explicit consumed species (one per repeat modified, ADP produced) and enters every rate as ATP/(K_ATP + ATP), with K_ATP fixed at 50 uM.
> ATP is a required entity, and the note states one ATP per phosphorylation. At most 800 uM of the 3000 uM is consumed, so with any K_ATP in the tens of uM the factor stays near 0.98 throughout. ATP is saturating, and its depletion cannot shape the data. K_ATP is therefore fixed, not estimated: no part of the data window constrains it. Mass action in ATP was rejected because it would impose a ~27% rate decline by 16 h as an artefact of the rate-law form.

**Alternatives considered.** Mass action in ATP (artefactual slowdown). ATP held constant as a parameter (loses the required entity and the mass balance). Estimating K_ATP (flat direction).

*confidence: high · evidence tier: default*
::::

::::{dropdown} P-TEFb is a constant species at 1 (relative units). Its absolute level is absorbed into kmod; CTD is in uM with CTD_0(0) = 100 uM.
> The enzyme concentration is not given, and only the product kmod\*PTEFb is visible. Fixing PTEFb = 1 removes that flat direction. Total substrate is known, so the model's amounts are absolute in uM. The arbitrary MS scale belongs to the observation model's per-time-point factors and does not trade off against any model state. Each repeat is one bit, so 9 count states are complete.

**Alternatives considered.** Estimating PTEFb and kmod separately (structurally non-identifiable). Tracking Ser2 and Ser5 separately (excluded by the note).

*confidence: high · evidence tier: direct*
::::

::::{dropdown} Not in the base, and named for the evaluation as what these data are expected not to separate: enzyme activity loss over 16 h, substrate saturation of the …
> Loss of kinase activity over 16 h would slow late modification exactly as negative cooperativity (cp \< 1) does at high n, and only the 960 min point sees either. Substrate saturation is a time reparameterisation (see the distributive-steps decision). Positive neighbour effects, positive count cooperativity and processivity all produce nucleation-then-fast-completion from counts at 3 time points without replicates. These are recorded so that a criterion difference among them is reported as an indistinguishable set, not as a mechanism. None is added to the base: no observable in the single condition is argued to separate them.

**Alternatives considered.** Adding a first-order enzyme inactivation state (deferred: a revision to argue from residuals at 960 min, not a priori).

*confidence: medium · evidence tier: default*
::::

::::{dropdown} No observation quantities in the Antimony. Readout phos\<n>_rel maps onto species CTD_n (n = 0..8) with one scale factor per time point, shared by the nine …
> The intensities are relative and drift between injections, so a per-time-point scale is a property of the measurement, not of the chemistry. The constant total substrate (100 uM) gives the dynamics the absolute reference, so the scales are identifiable from the shape of each spectrum.

**Alternatives considered.** Normalising states to fractions inside the model (rejected: it duplicates the observation layer).

*confidence: high · evidence tier: direct*
::::

::::{dropdown} The base model in Antimony (first 60 lines)
````text
model ctd_ptefb_counts
  // CTD_n: CTD molecules with n of 8 repeats phosphorylated (one bit per repeat)
  compartment vial = 1;
  species CTD_0 in vial, CTD_1 in vial, CTD_2 in vial, CTD_3 in vial, CTD_4 in vial;
  species CTD_5 in vial, CTD_6 in vial, CTD_7 in vial, CTD_8 in vial;
  species ATP in vial, ADP in vial, PTEFb in vial;
  CTD_0 = 100; CTD_1 = 0; CTD_2 = 0; CTD_3 = 0; CTD_4 = 0;
  CTD_5 = 0; CTD_6 = 0; CTD_7 = 0; CTD_8 = 0;
  ATP = 3000; ADP = 0; PTEFb = 1;
  kmod = 0.01; cp = 1; q_proc = 0; K_ATP = 50;
  // single-repeat encounters: one modification per binding event (distributive)
  J0: CTD_0 + ATP -> CTD_1 + ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*8*(1 - q_proc)*CTD_0;
  J1: CTD_1 + ATP -> CTD_2 + ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*7*cp*(1 - q_proc)*CTD_1;
  J2: CTD_2 + ATP -> CTD_3 + ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*6*cp^2*(1 - q_proc)*CTD_2;
  J3: CTD_3 + ATP -> CTD_4 + ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*5*cp^3*(1 - q_proc)*CTD_3;
  J4: CTD_4 + ATP -> CTD_5 + ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*4*cp^4*(1 - q_proc)*CTD_4;
  J5: CTD_5 + ATP -> CTD_6 + ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*3*cp^5*(1 - q_proc)*CTD_5;
  J6: CTD_6 + ATP -> CTD_7 + ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*2*cp^6*(1 - q_proc)*CTD_6;
  J7: CTD_7 + ATP -> CTD_8 + ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*cp^7*CTD_7;
  // processive encounters: j repeats per binding event, continuation probability q_proc (0 in base)
  Jp0_2: CTD_0 + 2 ATP -> CTD_2 + 2 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*8*q_proc*(1 - q_proc)*CTD_0;
  Jp0_3: CTD_0 + 3 ATP -> CTD_3 + 3 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*8*q_proc^2*(1 - q_proc)*CTD_0;
  Jp0_4: CTD_0 + 4 ATP -> CTD_4 + 4 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*8*q_proc^3*(1 - q_proc)*CTD_0;
  Jp0_5: CTD_0 + 5 ATP -> CTD_5 + 5 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*8*q_proc^4*(1 - q_proc)*CTD_0;
  Jp0_6: CTD_0 + 6 ATP -> CTD_6 + 6 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*8*q_proc^5*(1 - q_proc)*CTD_0;
  Jp0_7: CTD_0 + 7 ATP -> CTD_7 + 7 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*8*q_proc^6*(1 - q_proc)*CTD_0;
  Jp0_8: CTD_0 + 8 ATP -> CTD_8 + 8 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*8*q_proc^7*CTD_0;
  Jp1_2: CTD_1 + 2 ATP -> CTD_3 + 2 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*7*cp*q_proc*(1 - q_proc)*CTD_1;
  Jp1_3: CTD_1 + 3 ATP -> CTD_4 + 3 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*7*cp*q_proc^2*(1 - q_proc)*CTD_1;
  Jp1_4: CTD_1 + 4 ATP -> CTD_5 + 4 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*7*cp*q_proc^3*(1 - q_proc)*CTD_1;
  Jp1_5: CTD_1 + 5 ATP -> CTD_6 + 5 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*7*cp*q_proc^4*(1 - q_proc)*CTD_1;
  Jp1_6: CTD_1 + 6 ATP -> CTD_7 + 6 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*7*cp*q_proc^5*(1 - q_proc)*CTD_1;
  Jp1_7: CTD_1 + 7 ATP -> CTD_8 + 7 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*7*cp*q_proc^6*CTD_1;
  Jp2_2: CTD_2 + 2 ATP -> CTD_4 + 2 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*6*cp^2*q_proc*(1 - q_proc)*CTD_2;
  Jp2_3: CTD_2 + 3 ATP -> CTD_5 + 3 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*6*cp^2*q_proc^2*(1 - q_proc)*CTD_2;
  Jp2_4: CTD_2 + 4 ATP -> CTD_6 + 4 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*6*cp^2*q_proc^3*(1 - q_proc)*CTD_2;
  Jp2_5: CTD_2 + 5 ATP -> CTD_7 + 5 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*6*cp^2*q_proc^4*(1 - q_proc)*CTD_2;
  Jp2_6: CTD_2 + 6 ATP -> CTD_8 + 6 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*6*cp^2*q_proc^5*CTD_2;
  Jp3_2: CTD_3 + 2 ATP -> CTD_5 + 2 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*5*cp^3*q_proc*(1 - q_proc)*CTD_3;
  Jp3_3: CTD_3 + 3 ATP -> CTD_6 + 3 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*5*cp^3*q_proc^2*(1 - q_proc)*CTD_3;
  Jp3_4: CTD_3 + 4 ATP -> CTD_7 + 4 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*5*cp^3*q_proc^3*(1 - q_proc)*CTD_3;
  Jp3_5: CTD_3 + 5 ATP -> CTD_8 + 5 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*5*cp^3*q_proc^4*CTD_3;
  Jp4_2: CTD_4 + 2 ATP -> CTD_6 + 2 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*4*cp^4*q_proc*(1 - q_proc)*CTD_4;
  Jp4_3: CTD_4 + 3 ATP -> CTD_7 + 3 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*4*cp^4*q_proc^2*(1 - q_proc)*CTD_4;
  Jp4_4: CTD_4 + 4 ATP -> CTD_8 + 4 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*4*cp^4*q_proc^3*CTD_4;
  Jp5_2: CTD_5 + 2 ATP -> CTD_7 + 2 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*3*cp^5*q_proc*(1 - q_proc)*CTD_5;
  Jp5_3: CTD_5 + 3 ATP -> CTD_8 + 3 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*3*cp^5*q_proc^2*CTD_5;
  Jp6_2: CTD_6 + 2 ATP -> CTD_8 + 2 ADP; vial*kmod*PTEFb*ATP/(K_ATP + ATP)*2*cp^6*q_proc*CTD_6;
end
````
::::

### 3.3 · S1 — the candidates, and a stop on the way (`expansion` agent)

A second agent, the expansion agent ([the full prompt](prompts/expansion.md)), reads the base
and the intent and declares each candidate as a **difference from the base**:
which switch it turns on, or what it adds.

| candidate | what it claims | what changes from the base |
|---|---|---|
| **M1_independent** | Independent repeats, strictly distributive (base) | — (the base) |
| **M2_count_coop** | Count cooperativity between repeats (cp estimated) | `cp` estimated in [0.2, 5.0] |
| **M3_neighbour** | Nearest-neighbour effect (positional model, nb estimated) | adds `nb` |
| **M4_processive** | Processive continuation (q_proc estimated) | `q_proc` estimated in [0.0001, 0.99] |
| **M5_neighbour_and_count** | Nearest-neighbour effect plus count cooperativity (nb and cp estimated) | `cp` estimated in [0.2, 5.0], adds `nb` |

Getting the two positional candidates (M3 and M5) right took four calls and one
stop of the run — every step of it is in the record:

1. **The first draft was refused by a deterministic check** and sent back
   automatically: the patches for M3 and M5 used the generated positional states
   (the "gap classes") without declaring them in the base or in `adds_parameters`
   — 16 findings, such as
   > M3_neighbour: a patch uses symbol(s) \['Gf8'] that neither the base nor adds_parameters declares
2. **The second draft declared them**, as new *parameters*. It passed the S1
   checks, and the person approved the scope.
3. **At assembly (S3), `petablint` refused the estimation problem** — states
   declared as parameters are extraneous parameters to it:
   > petablint: Extraneous parameter(s) in parameter table: {'G02_11', 'G01_111', 'G14_1', 'G03_1', 'G01_22', 'G01_3', 'G01_4', 'G12', 'G00_5', 'G13', 'G34', 'G02_2', 'G01_21', 'G01_5', 'G01_11', 'G11_2', 'G02', 'G02_21', 'G12_1', 'G05', 'G12_3', 'G00_4', 'G02_3', 'G11_11', 'G03_3', 'G11_1', 'G00_41', 'G …

   No agent at that module could be asked to repair it, so the run asked whether to grant another
   attempt or stop, and the person answered **no**:
   the run ended there.
4. **The person re-entered the same run at the model module**
   (`--from-module model`): nothing upstream was paid for again. At the scope card
   they sent the expansion agent back with their own words:
   > redraft: In M3_neighbour and M5_neighbour_and_count, the 74 gap-class states (Gf8, G07 ... G00) are already declared as species in the append block; remove them from adds_parameters and keep only nb there. petablint rejected them as extraneous parameters in the parameter table. Change nothing else.
5. The third reply repeated the undeclared-symbol error and was sent back
   automatically; **the fourth passed**, and the person approved the scope again.


::::{dropdown} Five candidates. M1 is the base: independent repeats, strictly distributive. M2 switches on count cooperativity (cp estimated). M3 adds the nearest-neighbour …
> The intent asks for a three-way test (neighbour effect, count cooperativity, nothing at all) and a check of the distributive-versus-processive conclusion. M1 is the 'nothing at all' and distributive reference that every 'required' verdict is measured against. M2, M3 and M4 each switch on exactly one feature. M5 is the model that contains M2 and M3 as nested cases, so the comparisons 'neighbour effect beyond count cooperativity' and the converse are direct, and the joint nb-cp profile is where the separability question is answered. Every switch acts on its own: cp, nb and q_proc each change the dynamics with the others at their neutral values. No candidate differs from another only in an inert switch.

**Alternatives considered.** A processive-plus-cooperative combination (q_proc with cp), processivity on the positional model, and a free rate per count level (7 parameters, not identifiable from 9 peaks at 3 informative time points). The first two were left out to stay within five candidates, keeping the combination the question names directly (neighbour against count).

*confidence: high · evidence tier: direct*
::::

::::{dropdown} The neighbour effect, which the base cannot express, is added as a patch in M3 and M5. The patch implements construction's positional rule exactly: rate per …
> An unmodified repeat's rate depends only on n and on whether its two neighbours are marked. That is fixed by its position inside the maximal stretch of unmarked repeats (gap) that contains it, and by the gap's type: interior (marks on both sides), end (chain terminus on one side), or the whole unmodified chain. Modifying a repeat changes only its own gap: it splits the gap into at most two gaps of determined types, or shrinks it by one. The multiset of gap lengths with their types is therefore a sufficient state. The lengths of marked runs, the order of interior gaps and the orientation of the chain do not enter any rate; the rule treats both termini alike. Configurations sharing a gap structure have identical exit rates to every target class, so the 256 configurations are strongly lumpable onto the 74 classes. The classes with n marks sum to the count distribution exactly, at every nb and cp. Built-in check: at nb = 1 every class with n marks has total exit coefficient 8 - n (for example G01_22: 5\*nb), so M3 reproduces M1 and M5 reproduces M2 trajectory for trajectory. Writing the reduced network by hand carries far less risk of an enumeration error than 1024 hand-written lines.

**Alternatives considered.** The literal 256-species, 1024-reaction enumeration. The note prefers the positional model written out rather than avoided, and it is written out here. The enumeration would only split each class into configurations with identical trajectories, so it gives the same likelihood and the same verdicts; it is not a different model. A positional readout (which repeats are marked) would need it, and none exists.

*confidence: high · evidence tier: direct*
::::

::::{dropdown} In M3 and M5 the class reactions carry the count bookkeeping. Each G reaction from a class with n marks also converts CTD_n + ATP -> CTD_n+1 + ADP, at the same …
> This keeps CTD_n as the readout species shared by all five candidates, and it keeps ATP consumed exactly once per modification. Every flux into or out of a class with n marks moves CTD_n by the same amount, and CTD_n(0) equals its class sum, so CTD_n equals the sum of its classes identically. That identity now holds by stoichiometry, not by matching summed coefficients, so an error in any one coefficient cannot break it. A linear multistep integrator preserves the linear invariant to solver precision. The class reactions are the base's distributive single-repeat steps resolved by position, so J0 to J7 at zero rate are superseded, not contradicted. The base's processive Jp\* reactions likewise sit at exactly zero rate in M3 and M5, where q_proc stays 0 and processivity is not composed with the positional model. The class rate laws read only the class amount, so listing CTD_n as a reactant changes no rate. The deliberate simplification is a second conservation class (sum over G = 100 uM) alongside sum over CTD_n = 100 uM. Both are intended and neither carries a free parameter.

**Alternatives considered.** The previous structure, in which the J0 to J7 rates were rewritten as the summed class fluxes. It is mathematically equivalent, but the engine's symbol check refused it twice: a replacement line may not use species declared in another patch (attempts 1 and 2). Assignment rules CTD_n := sum of G_n were also rejected. They would require removing CTD_n from all 36 base reactions, which the patch grammar does not allow.

*confidence: high · evidence tier: direct*
::::

::::{dropdown} nb is estimated on log10 between 0.01 and 100, nominal 1 (neutral), as construction specified. cp in M2 and M5 keeps the base bounds 0.2 to 5. q_proc in M4 is …
> nb enters as nb^2 on a repeat whose two neighbours are both marked, so its bounds span 8 decades on those steps. Together with cp^7, the fastest corner reaches rates about 1e4-fold above the per-repeat kmod. The engine should check integrability at the corners of the box, not only at the nominals (general-dynamics pack entry 8). The q_proc lower bound is strictly distributive to within 1e-4. If q_proc rests on that bound after fitting, that is a finding in favour of distributive action, not a bounds artefact (pack entry 6). Because M1 sits on the boundary of M4, a likelihood-ratio reading of M4 against M1 needs the boundary correction. The q_proc profile is the direct test.

**Alternatives considered.** nb bounds of 0.1 to 10. These would exclude the strong-inhibition regime (jamming at alternate repeats) that gives nb its most distinctive signature at 16 h.

*confidence: medium · evidence tier: default*
::::

::::{dropdown} Expected indistinguishable set and unresolved questions, named for the evaluation. nb > 1, cp > 1 and q_proc > 0 all predict nucleation then fast completion, …
> These data resolve count distributions, not positions. Neighbour inhibition caps fast modification at 4 marks on 8 repeats (alternate repeats) before a slow filling phase, a shape cp alone cannot produce. Neighbour facilitation, by contrast, aliases with count cooperativity and with processivity. M5's joint profile will show whether nb and cp are separately determined or only their combined effect. If the nb-cp profile is flat along a ridge, 'neighbour effect versus count cooperativity' must be reported as unresolved, with the combined effect reported as determined. Sliding processivity is positional and is likely aliased with nb > 1. Kinase inactivation over 16 h aliases with cp \< 1, since only the 960 min point sees either. Substrate saturation reparameterises time. All of these follow construction's decisions and none is added this round.

**Alternatives considered.** Adding kinase inactivation as a sixth hypothesis. It exceeds the five-candidate cap, and no observable in the single condition is argued to separate it from cp \< 1.

*confidence: medium · evidence tier: to_be_estimated*
::::

::::{dropdown} Revision on human feedback and the gate errors. In M3_neighbour and M5_neighbour_and_count, adds_parameters holds only nb, as asked. The 74 gap classes (Gf8, …
> I could not satisfy 'change nothing else' and pass the gate at the same time. The previous reply already did exactly what was asked, and the gate refused M3 and M5 for the reason it gave in attempt 1. A replace_reaction line may use only symbols from the base or adds_parameters, and putting the classes in adds_parameters is what petablint rejected. The closest defensible change touches only the lines the gate names (the eight J replacements) and moves their job, moving CTD_n and consuming ATP, into the class reactions. The model is the same: every species trajectory, and therefore the likelihood, matches the previous structure, because CTD_n equals the class sum in both. The gate never flagged the append block, even though it uses every G symbol. That suggests each patch is checked against its own declarations plus the base and adds_parameters, and not against other patches. The replacement lines now use base symbols only, so they pass under either reading.

**Alternatives considered.** Reordering the patches so the append comes first. Not chosen: the append's acceptance suggests the check is per patch, not cumulative, so the order would probably not matter. Putting the classes back in adds_parameters: rejected, because petablint refuses them. Changing the engine's symbol check to accept species declared in another patch of the same candidate: that would let the previous structure stand and is the engine maintainers' call, but this reply does not depend on it.

*confidence: high · evidence tier: direct*
::::

### 3.4 · S1 — checking the model

Two kinds of check run before anyone is asked. **Deterministic gates** build and
simulate every candidate (a ✓ is a pass, ⚠ a finding):

| check | M1 | M2 | M3 | M4 | M5 |
|---|---|---|---|---|---|
| `model_antimony_builds` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `model_conservation_laws` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `model_declared_constants` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `model_estimated_parameters_used` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `model_no_dynamics_screen` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `model_nonnegative_states` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `model_refs_resolve` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `model_required_entities` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `model_sbml_valid` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `model_simulates_across_box` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `model_simulates_nominal` | ✓ | ✓ | ✓ | ✓ | ✓ |

- every check passed for every candidate

Then an **independent checker agent** ([the full prompt](prompts/construction_verification.md)) audits the
construction's reasoning against the note — its citations, its support, its
rationales:


**The checker's overall call: `concerns`** — 3 of 6 aspects fine, the rest raised as concerns (click an aspect for its note):

::::{dropdown} ⚠ concern · citation honesty
> No external literature is cited and no entry is marked resolution: unresolvable. The general-knowledge statements are accurate: strong lumpability (Kemeny-Snell), mass action as the low-saturation limit of Michaelis-Menten, and protein-kinase ATP Km of 10-100 uM. Several entries are mis-sourced, though. (1) Decision 10 (five candidates) evidence 2, decision 11 (positional patch) evidence 3 and decision 14 (indistinguishable set) evidence 2 give source 'lab_note', but their locators ('base_model decisions (count cooperativity, processivity switches)', 'base_model decision: exact construction rule...', 'base_model decision: not in the base...') are the build report's own decisions 2, 4, 5 and 8; the lab note contains no such text. (2) Decisions 12, 13 and 15 file engine error logs ('previous_attempt_errors', 'attempt_history'), domain-pack entries and human feedback (petablint rejection) under source 'general_knowledge'. The origins ('inferred', 'user-provided') show nothing is invented, but the source fields should name the build report, the engine log, the pack and the human feedback so a reader can check them.
::::

::::{dropdown} ✓ support
> Each lab-note quote matches its decision. Purified components only supports no phosphatase and irreversibility (decision 3). Catalytic and constant kinase supports PTEFb as a fixed modifier (decision 7). 3 mM ATP with 100 uM x 8 repeats supports the at-most-800 uM consumption argument and K_ATP being immaterial (decision 6). One bit per repeat supports the 9 count states. The relative, drifting intensities support per-time-point scales in the observation layer (decision 9). The three-hypothesis passage supports cp, nb and the five-candidate design. The general-knowledge entries back only modelling-form choices (lumpability, the mass-action limit, the continuation-probability representation of processivity) and are recorded as assumed or inferred. No correlational or adjacent material is recorded as mechanism.
::::

::::{dropdown} ⚠ concern · rationale quality
> Most rationales argue from the system: ATP stays saturating (factor about 0.98); 0.2-5 on cp compounds to about 10 decades at n = 7; neighbour inhibition jams at 3-4 non-adjacent marks; processivity's memory distinguishes it from cp > 1. Three specific problems. (a) Decision 13 says that with cp^7 'the fastest corner reaches rates about 1e4-fold above the per-repeat kmod'. In M5 at cp = 5, nb = 100, Q_G00_1_G00 carries cp^7\*nb^2 = 78125\*1e4, about 7.8e8. The slow corner (cp = 0.2, nb = 0.01) is about 1.3e-9, so the box spans about 18 decades on that step, not 8. The 1e4 holds only for M3 (cp = 1). The corner-integrability check the decision asks for should use the correct span. (b) Decisions 1 and 2 are stale relative to decision 11. Decision 1 says 'The positional (neighbour) model is not written in this reply' and still carries user_decision_required: true with options A (generate 1024 reactions mechanically) and B (hand-enumerate 256 states). What was delivered is neither: a hand-written exact 74-class lumping. The human at the scope gate is asked a question that no longer describes the record. Part of decision 1's rationale is also about the builder's capacity ('Two previous replies ... failed the contract') rather than the system. (c) Decision 11 says the full enumeration 'would only split each class into configurations with identical trajectories'. That is false for nb != 1. For example, 10110000 and 11010000 are both in G04_1 but receive different inflows: 10110000 gets nb from 10100000, nb from 10010000 and 1 from 00110000; 11010000 gets 1 from 11000000, nb from 10010000 and nb from 01010000. Only the class sums are exact. The conclusion (same likelihood and verdicts) still holds.
::::

::::{dropdown} ✓ tier honesty
> The default labels are appropriate: decision 3 (mass action in CTD, irreversibility), decision 6 (K_ATP fixed at 50 uM from a generic range, argued immaterial because ATP >> K_ATP throughout), decision 1 (count-level base) and decision 13 (bounds). cp, nb and q_proc are correctly to_be_estimated. 'direct' on decision 7 (CTD_0(0) = 100 uM, constant kinase), decision 9 (per-time-point scaling) and the J0-J7 stoichiometry rests on lab-note statements. 'direct'/high on decisions 11, 12 and 15 is generous: they rest on a lumping derivation and engine workarounds, not system evidence. The lumping was verified under structure sanity, so no mechanistic support is overstated. Nothing labelled default has unclaimed support.
::::

::::{dropdown} ✓ structure sanity
> The base Antimony matches the network: 12 species, and J0-J7 plus 28 Jp reactions (36 in total) with j ATP consumed per j-repeat run. The per-repeat encounter rate is (8-n)\*cp^n. Run weights are q^(j-1)(1-q), with q^(r-1) for the run that completes the molecule, so J7 correctly lacks (1-q_proc) and Jp6_2 = q_proc; each encounter's weights sum to 1. The M3/M5 patch was checked class by class against decision 2's rule. The 74 gap classes are exactly the gap structures with n = 0..8 marks (1, 4, 16, 20, 16, 9, 5, 2, 1), and the 257 reactions have the correct target classes. nb exponents equal the number of marked immediate neighbours, including terminal repeats, and cp powers equal n. Each class's coefficients sum to 8-n at nb = 1 (for example G04_1: nb^2+nb+3; G01_22: 5nb; G00_111: 3nb^2). Each Q reaction also moves CTD_n to CTD_n+1 and consumes one ATP, so CTD_n equals its class sum identically. J0-J7 at zero rate and the Jp reactions at q_proc = 0 contribute nothing in M3/M5. Minor gap: kmod's nominal value (0.01 per min) and its estimation scale and bounds are not recorded in any decision.
::::

::::{dropdown} ⚠ concern · candidate set
> The reference model M1 (cp = 1, q_proc = 0, no positional term) is present, as both the three-way test and the distributive test need. M2 (cp), M3 (nb, cp held at 1) and M4 (q_proc) each switch on one feature. M5 declares its nb+cp combination and carries the identical positional patch, which reduces to M2 at nb = 1 and to M3 at cp = 1. The J0-J7 zeroing and Q-reaction bookkeeping change no trajectory at neutral values, and kmod and K_ATP are handled identically across all five, so no candidate differs outside its named feature. The concern is M4. It implements random-site continuation: the kinase picks any remaining unmodified repeat. The lab note defines processive as 'holding on and running along the substrate', and asks that possibilities representable only positionally be fitted rather than settled from counts. Sliding processivity was set aside in decisions 5, 10 and 14 as 'likely aliased with nb > 1' to stay within five candidates, which is an argument from counts. M4's decision rule ('If it reaches the lower bound ... distributive action is supported') therefore overstates. A q_proc at its lower bound would exclude only random-site processivity. Either the M4 verdict should be reported as testing random-site processivity only, with sliding listed as untested, or the human should decide whether a positional sliding candidate replaces M5 or goes beyond the cap.
::::

The checker's findings go on the scope card; they do not block the run. The
last scope card of round 0, with the answer that let the candidates go on:

:::{admonition} A gate · model scope (round 0)
:class: important
**The card asked:** Discriminate between these 5 candidate mechanisms? (each will be compiled and fitted)

**The answer:** **yes**, given at the card.

**The note:**

> 1: M3 and M5 as built (the 74 exact gap classes) are the neighbour test; no base redraft
:::

### 3.5 · S2 — reading the data (`data_extraction` agent)

The data agent ([the full prompt](prompts/data_extraction.md)) maps the CSV onto measurements —
9 observables, one per phosphorylation count — and raises
**flags** about what it saw, which travel to every later agent:


::::{dropdown} design_summary role guess is WRONG and is corrected here:
> design_summary role guess is WRONG and is corrected here: phos1_rel..phos8_rel are measurement columns (relative MS intensities for 1..8 phosphates), not design factors; their 'levels' are measured values, and the 2916-cell 'factorial' is meaningless. The file has NO condition columns: one in vitro condition (8-repeat CTD 100 uM, P-TEFb catalytic, ATP 3 mM) sampled at 4 times. All 9 phosN_rel columns are mapped as separate observables of that single condition; condition_columns is empty and the label template is a fixed string.
::::

::::{dropdown} ctd_ms_timecourse.csv, all columns phos0_rel..phos8_rel, all 4 rows:
> ctd_ms_timecourse.csv, all columns phos0_rel..phos8_rel, all 4 rows: intensities are on a per-injection arbitrary scale that differs between time points (row sums 1.00 at 0 min, 4.07 at 60 min, 3.79 at 120 min, 0.85 at 960 min, matching the lab note) while total substrate is conserved. Values are NOT commensurable across time points as they stand. No processing was declared: the note does not say the data were normalised, and the note's own observation model (a per-time-point factor mapping amounts onto intensities, which also scales the noise SD) belongs model-side. Do not pool or compare raw values across rows.
::::

::::{dropdown} ctd_ms_timecourse.csv:
> ctd_ms_timecourse.csv: 19 of 36 measured values are exactly 0.0 (row time_min=0: phos1_rel..phos8_rel; row 60: phos7_rel, phos8_rel; row 120: phos8_rel; row 960: phos0_rel..phos7_rel). The note states additive Gaussian noise of a single SD at every peak, under which a measured empty peak would scatter around zero (including small negatives), not sit at exactly 0.0. These exact zeros look like not-detected or absent peaks written as zero, or idealised values, but the source states no detection floor and no handling rule, so no below_detection entry is declared. Left as is and NOT excluded; the observation model should know that roughly half the likelihood terms are exact zeros of uncertain provenance.
::::

::::{dropdown} No replicates (n = 1 per species per time point, stated in the note) and no error column.
> No replicates (n = 1 per species per time point, stated in the note) and no error column. error_type recorded as 'none' because the file carries no error estimate. The note describes the noise model (additive, normal, one SD for the whole experiment shared by all peaks, scaled per time point by the same factor that maps amounts onto intensities); that SD is not given and must be estimated by the observation model. This is not an SD/SEM call: no error bars exist.
::::

::::{dropdown} Information content is thin for mechanism discrimination.
> Information content is thin for mechanism discrimination. Of 4 time points, t = 0 carries only the initial condition (all substrate unmodified; phos0_rel = 0.99957). t = 960 min carries only completion (only phos8_rel nonzero). Effectively only the 60 and 120 min distributions carry shape information about rates, each losing one degree of freedom to its own scale factor. The readout is phosphate COUNT only (9 mass peaks); which repeats are modified is not observed, so position-dependent (neighbour) effects can only enter through their imprint on count distributions.
::::

::::{dropdown} time_min = 960 is a sampled time (16 h), not a declared steady-state measurement, and is mapped as 960, not inf.
> time_min = 960 is a sampled time (16 h), not a declared steady-state measurement, and is mapped as 960, not inf. It appears to be at or near completion (all signal in phos8_rel), but treating it as steady state would be a modelling decision, not an extraction edit.
::::

::::{dropdown} Time units are minutes, consistent with the intent's horizon (960 min).
> Time units are minutes, consistent with the intent's horizon (960 min). No conversion was made.
::::

### 3.6 · S3 — the observation model

The instrument does not measure concentrations. The observation-model agent
([the full prompt](prompts/observation.md)) decides how each state maps onto a measured
intensity:

- **Formula:** intensity of the *k*-phosphate peak = `sc_ctd_ms` (one per time point) × the amount of CTD with *k* modified repeats
- **Noise:** `normal`, sd = `sc_ctd_ms * sigma_ctd_ms` — one standard deviation for the whole experiment, scaled with each spectrum, as the note says
- **Scaling:** `factor` per `timepoint`, shared by the nine peaks of one injection


::::{dropdown} The agent's rationale for the observation model
> The readout is intact-mass MS of the 8-heptarepeat CTD construct (the lab note names it). The phos0 peak is the unmodified mass, which is exactly the model's lumped species CTD_0 (glossary: the single configuration with no marks). Species are in uM with vial = 1, so no volume factor is needed. If a position-resolved candidate is fitted, this label maps to its single zero-mark configuration. Units are relative intensities on a per-injection scale (data_labels: arbitrary). The lab note shows spectra summing to 1.00, 4.07, 3.79 and 0.85 at 0, 60, 120 and 960 min while substrate is conserved. A scale factor is therefore required, indexed per time point and shared by all nine peaks under group ctd_ms, because one injection gives one spectrum and one gain. No offset: no background, blank or instrument floor is described, and the exact zeros in the table argue against an unsubtracted baseline. The absolute scale is pinned by the fixed model total: CTD_0 starts at 100 uM, the total over CTD_0..CTD_8 is conserved, and that amount must stay unestimated. Pinning the first factor to 1 would instead clash with the t=0 spectrum summing to 1 against 100 uM. The factors are thus intensity per uM, roughly 0.010, 0.041, 0.038 and 0.0085, consistent with the nominal-magnitude gap. Noise: the lab note states additive normal error, one SD for the whole experiment, equal for every peak, scaling between time points with the same factor that maps amounts onto intensities. That is the gain-scaled shape sc_ctd_ms \* sigma_ctd_ms, with one sigma_ctd_ms shared by name across all nine observables. Per-peak sigmas would rest on 4 points each and collapse toward zero on peaks the model fits exactly. data_signals is inconclusive (n = 1, no replicates), so this choice follows the lab note rather than a replicate test. The scale stays lin because this peak is zero once (log illegal) and the error is stated to be additive. An assumption that cannot be tested here: every phosphorylation count has the same ionisation and detection response. A count-dependent response factor would be confounded with the cooperativity parameter cp, so it is named rather than estimated. Evidence: direct (the lab note names the assay, the units and the error model).
::::

::::{dropdown} The condition encoding (`condition` agent)
````json
{
 "conditions": [
  {
   "condition_id": "cond_invitro_PTEFb_CTD8",
   "maps_to_label": "invitro_PTEFb_CTD8",
   "overrides": {
    "CTD_0": 100.0,
    "CTD_1": 0.0,
    "CTD_2": 0.0,
    "CTD_3": 0.0,
    "CTD_4": 0.0,
    "CTD_5": 0.0,
    "CTD_6": 0.0,
    "CTD_7": 0.0,
    "CTD_8": 0.0,
    "ATP": 3000.0,
    "ADP": 0.0,
    "PTEFb": 1.0
   },
   "preequilibration": false,
   "rationale": "The intent lists no perturbation. The data hold a single label: one purified in vitro P-TEFb kinase reaction, sampled at 0, 60, 120 and 960 min with no replicates. That makes exactly one condition, and none is invented. The overrides only restate the reaction mix as the lab note gives it (tier direct: the note states amounts and sampling times), so the audit can see what this condition was. These values match the model's initial values, so they change nothing in the simulation. CTD_0 = 100 uM, the recombinant 8-heptarepeat substrate, all unmodified at the start of the reaction. CTD_1..CTD_8 = 0 uM, since no modified substrate exists at t = 0. ATP = 3000 uM (3 mM at start, model unit uM, so there is no unit slip). ADP = 0 uM, as no phosphoryl transfer has occurred yet. PTEFb = 1 relative unit: the glossary says it is catalytic and constant, with its absolute concentration absorbed into kmod; the note gives no kinase concentration, so 1 is a convention, not a measured dose. No assignment rules or initial assignments exist, so these initial-value overrides are not silently ignored. The CTD_0 = 100 uM total is what pins the absolute scale in the observation draft. Its per-time-point gains (intensity per uM) are identified only because this total is fixed and unestimated. Estimating it would make it degenerate with the scale factors. Likewise, kmod absorbs kinase amount, so PTEFb must stay fixed. No parameter is overridden. kmod, cp and q_proc are mechanism parameters to be estimated or switched by model selection, not properties of this condition. The glossary marks none of them 'set per condition', which is expected for a single-condition design. K_ATP stays at its construction-fixed 50 uM. No population factor exists, so nothing is split per group. Pre-equilibration is false. t = 0 is the moment kinase, substrate and ATP are combined, not a settled baseline. basal_state confirms that the model's only steady state is full modification (CTD_8), so pre-equilibrating would run the reaction to completion before the first sample. Correction to design_summary: phos1_rel..phos8_rel were guessed as 'design' factors, but they are measurements (relative MS intensities for 1..8 phosphates), like phos0_rel. Their few unique values reflect only four rows with exact zeros, not factor levels. The 2916-cell 'factorial' is therefore spurious."
  }
 ]
}
````
::::

The assembled estimation problem (PEtab) is then checked for every candidate
(the final assembly, after the redraft above):

| check | M1 | M2 | M3 | M4 | M5 |
|---|---|---|---|---|---|
| `assembly_conditions_resolve` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_connected` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_dof` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_formulas_compile_safe` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_magnitude_reconciliation` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_measurements_resolve` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_noise_declared` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_observables_resolve` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_parameters_sane` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_petablint` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_preequilibration` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_sensitivity_coverage` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_sigma_through_observable` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `assembly_verification` | ✓ | · | · | · | · |

- every check passed for every candidate

and audited by a second checker agent ([the full prompt](prompts/assembly_verification.md)):


**The checker's overall call: `concerns`** — 4 of 6 aspects fine, the rest raised as concerns (click an aspect for its note):

::::{dropdown} ✓ noise model vs readout technique
> Checked against the lab note's stated error model for the intact-mass MS readout. Every observable uses a normal distribution on a lin scale. Its noise SD is observableParameter1 × noiseParameter1, which is the per-spectrum factor times the single sigma_ctd_ms. This matches the note's claims: additive Gaussian error, one SD for all peaks, scaled by the same factor that maps amounts to intensities at that time point. Reusing observableParameter1 in noiseFormula guarantees the noise and signal factors are the same parameter. The lin scale tolerates the 19 exact zeros, where a log transform would fail. No error column exists, so there is no SD/SEM question, and estimating sigma is the only option. Doubt only: MS peak noise often grows with intensity. The assembly follows the note's additive claim, which is defensible, but residuals at 60 and 120 min should be checked for intensity dependence.
::::

::::{dropdown} ⚠ concern · structurally exact likelihood terms under the shared estimated sigma
> Checked the t=0 and t=960 rows against the initial-condition encoding and the per-time-point scales. At t=0 every candidate mechanism reproduces all nine terms with zero residual by construction. CTD_1..CTD_8 start at exactly 0, matching the exact-zero data. The free sc_ctd_ms_t0 (best value about 0.01, inside its box) absorbs the single nonzero peak. At t=960 the free sc_ctd_ms_t960 likewise absorbs phos8. These terms carry no information about mechanism, but they still count in the likelihood and pull the shared sigma_ctd_ms estimate toward zero. With sigma estimated, likelihood differences between candidates scale with the total number of terms. That inflates the evidence separating mechanisms, which affects the Akaike weights the intent requires to be reported honestly, as well as profile-based intervals. The data flags asked the observation model to account for the exact zeros. Nothing in observables or parameters shows that this was considered. Point estimates of kmod, cp and q_proc are not affected; sigma and the model-selection evidence are.
::::

::::{dropdown} ✓ condition encoding vs experiment description
> Reconstructed the single treatment from its overrides alone: 100 µM CTD, all in CTD_0; ATP 3000 µM (3 mM); ADP 0; PTEFb 1. This matches the note's 8-repeat substrate at 100 µM, 3 mM ATP, a catalytic kinase that is never consumed, and t=0 data with all signal in phos0. One condition matches the note's one in vitro treatment sampled over time. No pre-equilibration is used, which is correct: there is no pre-treatment, and the reaction starts at mixing. Fixing PTEFb at 1 is consistent with the glossary's statement that kinase concentration is absorbed into kmod. Fixed K_ATP of 50 µM is harmless, since at most 800 µM ATP is consumed out of 3000.
::::

::::{dropdown} ✓ scaling and offset vs declared units
> Checked against the note's statement that intensities are arbitrary, drift between injections and are not comparable across time points. There are four per-time-point scale factors (t0, t60, t120, t960). This is per-spectrum scaling with an explicit stated reason, not unexplained per-condition scaling. No information is lost that the data ever held, because cross-time amounts are declared incomparable, and each time point keeps its within-spectrum shape. The factors are identifiable rather than flat: the model total is fixed and conserved at 100 µM, so each factor is pinned near row-sum/100 (about 0.0085 to 0.041), well inside the 1e-3 to 1e3 log10 box. There is no offset, which is appropriate: the note gives no background story, and the exact-zero empty peaks argue against one. Consequence to carry into the report: each informative spectrum (60, 120 min) gives up one degree of freedom to its factor.
::::

::::{dropdown} ✓ observable mapping vs what the assay measures
> Checked each mapping against the glossary lumps and the intact-mass readout. Each phosN observable maps to CTD_n, which lumps all C(8,n) configurations with n marks. That is exactly what one mass peak sees, since position is not resolved and the one-bit-per-repeat convention makes count equal phosphate number. No lumped state is omitted. ATP, ADP and PTEFb are correctly left unobserved, and no kinase-bound substrate species exists that would need adding to a total. I cannot verify from these artifacts that each measurement row uses the observable id matching its own time point; the ids hard-wire the time point. Doubt: the mapping assumes every phospho-form ionises with the same response factor within a spectrum, as the note's model also does. A count-dependent response would tilt peak heights systematically with n, in a way the count-cooperativity factor cp could absorb. This is a confound for the cooperativity conclusion that should be named in the report, not a fault in the table.
::::

::::{dropdown} ⚠ concern · hypothesis space vs the mechanistic question asked
> Checked the glossary species and the parameters table against the intent and the lab note. The note asks that neighbour-dependent modification be fitted rather than settled by argument, explicitly inviting the 256-configuration model. The assembly holds only count-indexed species CTD_0..CTD_8, plus switches for count cooperativity (cp) and processivity (q_proc). A rate that depends on whether a repeat's immediate neighbours are modified is not a function of CTD_n alone, so it cannot be expressed in this state space. No parameter for it exists. As assembled, the comparison covers only two of the three hypotheses the intent names, and Akaike weights would be computed over a set that omits the hypothesis the note most wants tested. Minor, related: q_proc's distributive null value of 0 lies outside its own log10 box (1e-4 to 0.99) and has no finite log10 value. The base model's fixed 0 on a log10 scale may need checking in the tooling, and the switched-on test compares against q_proc ≥ 1e-4 rather than exactly distributive.
::::

### 3.7 · Compile and budget — the cost gates

Compiling five AMICI models costs minutes, so the person is asked first. After
compiling, a short **cost probe** times a few optimiser starts per candidate and
projects what the full estimation would take; that projection is the budget card.

| candidate | median time per start (probe) | starts approved |
|---|---|---|
| M1_independent | 1.8 s | 100 |
| M2_count_coop | 2.1 s | 100 |
| M3_neighbour | 12.3 s | 100 |
| M4_processive | 2.0 s | 100 |
| M5_neighbour_and_count | 16.1 s | 100 |

The budget was approved on the projection for the first candidate. The engine
does not stretch an approval further than it was given: the positional models
M3_neighbour and M5_neighbour_and_count probed about 7 and 9 times slower than that,
beyond the declared ceiling of 5×, so each was put to the person on its own card
(3 budget cards in all).

:::{admonition} A gate · compile (round 0)
:class: important
**The card asked:** Approve AMICI compile + cost probe for 5 candidates?

**The answer:** **yes**, applied by the front end from the owner's standing answers.

**The note:**

> follows the owner's standing answers of 2026-10-04 (assembly_cost_boundary: y)
:::

:::{admonition} A gate · estimation budget (round 0)
:class: important
**The card asked:** Run the full estimation with 100 starts x 5 candidates (projected ~898s total)?

**The answer:** **yes**, applied by the front end from the owner's standing answers.

**The note:**

> follows the owner's standing answers of 2026-10-04 (budget: y, default starts, forward sensitivities)
:::

:::{admonition} A gate · estimation budget (round 0)
:class: important
**The card asked:** Run the full estimation for candidate 'M3_neighbour' with 100 starts (projected ~137s wall)? Its probe differs from the approved candidate's — see the first lines.

**The answer:** **yes**, applied by the front end from the owner's standing answers.

**The note:**

> follows the owner's standing answers of 2026-10-04 (budget: y, default starts, forward sensitivities)
:::

:::{admonition} A gate · estimation budget (round 0)
:class: important
**The card asked:** Run the full estimation for candidate 'M5_neighbour_and_count' with 100 starts (projected ~179s wall)? Its probe differs from the approved candidate's — see the first lines.

**The answer:** **yes**, applied by the front end from the owner's standing answers.

**The note:**

> follows the owner's standing answers of 2026-10-04 (budget: y, default starts, forward sensitivities)
:::

### 3.8 · S6 — fitting every candidate

Each candidate was fitted from 100 random starts. The **waterfall** shows the
final objective of every start, sorted: a flat plateau at the left means many
starts found the same best fit, so the optimum is trustworthy.

![Multistart waterfalls, round 0](figures/waterfalls_round0.svg)

| candidate | best −log L | `kmod` | `cp` | `nb` | `q_proc` | `sigma_ctd_ms` |
|---|---|---|---|---|---|---|
| **M1_independent** | -61.36 | 0.00639 | 1 |  | 0 | 2.41 |
| **M2_count_coop** | -81.64 | 0.0051 | 1.14 |  | 0 | 1.33 |
| **M3_neighbour** | -84.25 | 0.00489 | 1 | 1.81 | 0 | 1.23 |
| **M4_processive** | -80.95 | 0.00521 | 1 |  | 0.212 | 1.32 |
| **M5_neighbour_and_count** | -91.20 | 0.00458 | 1.59 | 0.314 | 0 | 1.01 |

At a bound in some fit: nothing.


### 3.9 · S7 — comparing the candidates

The engine ranks the candidates by AIC. Two models within ΔAIC 2 are called
**indistinguishable**; the selection is called decisive when the winner's Akaike
weight exceeds 0.70.

![Selection, round 0](figures/selection_round0.svg)

**M5_neighbour_and_count** wins decisively, with weight 0.997. Then the interpretation agent
([the full prompt](prompts/interpretation.md),
340,245 characters of fits,
residuals, attribution and diagnostics) reads the evidence and writes the answer.
Its recommendation: **`revise`** — the winner still misses
systematically, and the agent says where.


::::{dropdown} The agent's diagnosis, in full
> Within the five fitted candidates, M5_neighbour_and_count is selected decisively. Its AIC is -166.40, its Akaike weight 0.9971, and the indistinguishable set is {M5_neighbour_and_count}. The others trail by ΔAIC 11.90 (M3_neighbour), 17.12 (M2_count_coop), 18.50 (M4_processive) and 55.69 (M1_independent); petab_select agrees.
>
> OBSERVATION MODEL AS FITTED. Each of the 36 cells is y_n(t_j) = sc_ctd_ms_tj \* \[CTD with n phosphates](t_j) + e, with e ~ N(0, (sigma_ctd_ms \* sc_ctd_ms_tj)^2), on a linear scale. Each spectrum has one free scale: t0 0.009986, t60 0.03971, t120 0.03844 and t960 0.008519 per µM. These map 100 µM to 1.00/3.97/3.84/0.85, against measured sums of 1.00/4.07/3.79/0.85. One sigma_ctd_ms = 1.012 µM-equivalent is shared by every cell. Check: the phos5 1-h residual of 0.1027 equals 2.555 x 1.012 x 0.03971. Substrate conservation holds to 1e-13.
>
> WHY M5 WINS (attribution).
> - Against M1, the 1-h spectrum pays: phos1 3.27, phos3 2.63, phos0 2.29 and phos2 1.71 in -log L. An independent, homogeneous model must give a binomial, but the measured 1-h distribution is much broader. Measured phos0 is 0.394 against 0.183 predicted, and the variance is about 1.5x binomial at the same mean.
> - Against M2 and M3, the gain is in phos2 at 1 h (2.77, 2.13), phos4 at 1 h (1.49, 1.38) and phos7 at 2 h (1.32, 1.76). M5 gives back phos0 at 1 h and phos2/phos6 at 2 h.
> - Against M4, the gain is phos7 at 16 h (3.21; M4's slower base rate leaves 3% stuck at seven), phos1 at 1 h (1.70) and phos8 at 2 h (1.62; M4 predicts 0.145 where 0 was measured). M4 is better on the 1-h upper tail (phos5 -2.66, phos6 -1.73).
>
> The data demand a distribution that is broad at 1 h, has an empty phos8 peak at 2 h, and is complete at 16 h. M5 gets there by pairing count acceleration (cp 1.588) with nb 0.314. If nb is a per-neighbour multiplicative factor with neutral value 1, like cp, this brakes the filling of flanked gaps late in the reaction.
>
> TWO LIMITS ON THIS RESULT.
> (i) Roughly half of every margin is sigma bookkeeping, not mechanism. The 18 cells at 0 h and 16 h are fitted to within noise by every model (19 measured values are exact zeros). Each adds log(sigma_rival/sigma_M5) to the winner: 0.193 per cell against M3, about 3.5 of its 6.95 Δ(-log L). On the 18 informative cells at 1 h and 2 h, the margin over M3 is about ΔAIC 5. That is still above 2, but the weight is nearer 0.9 than 0.997. Exact zeros are also not what the stated additive Gaussian noise produces.
> (ii) The neighbour term is seen only through the count distribution, and its value moves across neutral between models (nb 1.812 in M3, 0.314 in M5). What it buys in M5 is a step-rate profile that rises through mid-counts and brakes at the end, which M2's single cp cannot express. No count-only candidate with an equally flexible step-rate profile was fitted. So this round does not show that the effect is specifically about immediate neighbours.
>
> REMAINING STRUCTURE. The per-series shape tools are degenerate here: each peak x time is its own observable with n = 1, so sign runs, autocorrelation and residual_forms carry no shape. Read across counts within each spectrum, M5's largest residuals form one pattern:
> - the 1-h upper tail is underpredicted (phos5 +2.6 sigma, 0.352 vs 0.250; phos6 +2.0 sigma);
> - the 2-h top is overpredicted (phos8 -2.3 sigma, 0 vs 0.088; phos7 -2.0 sigma);
> - the 2-h low tail is heavier than any candidate gives (phos1 at 2 h is 0.191, against at most 0.164).
> There are no replicates, sigma is fitted from these same residuals, and the noise check has n = 1 per cell. Absolute adequacy therefore cannot be judged; chi2/dof is about 1 by construction.
>
> IDENTIFIABILITY. The FIM spans 2.86 decades with no flat direction; its smallest eigenvector is sigma_ctd_ms. Profiles were not computed, so no intervals exist. The nested reductions still exceed the 1.92 profile threshold even if halved: M2 drops nb (Δ(-log L) 9.56) and M3 fixes cp = 1 (6.95). Both terms are therefore non-null within M5.
>
> ANSWERS TO THE INTENT.
> - 'The rate depends on nothing at all' is rejected for a homogeneous substrate. Static substrate heterogeneity was not tried.
> - Count-only and neighbour-only each fall short of both together. Whether the second term is specifically an immediate-neighbour effect is not established.
> - Distributive vs processive: processivity alone (q_proc 0.2124) loses by ΔAIC 18.5, which is consistent with the by-eye call. But processivity was never combined with the coupling terms, and M4 fits the 1-h upper tail best. 'Distributive' therefore holds only against processivity alone.
> - Global effects cannot produce the non-binomial shape. These are ATP (3 mM with at most 0.8 mM consumed, K_ATP fixed at 50), kinase loss and product inhibition. They multiply every step equally and only re-time the trajectory.
>
> REVISE, with one move: fit a count-only candidate with a three-parameter step-rate profile and compare its -log L with M5's -91.20.
::::

::::{dropdown} Its answer, criterion by criterion
**the selected mechanism is reported with its information criterion, its Akaike weight and the indistinguishable set, honestly, whatever they turn out to be** — `pass`

> M5_neighbour_and_count: AIC -166.40, Akaike weight 0.9971, indistinguishable set {M5_neighbour_and_count} at the ΔAIC threshold of 2. Next are M3 11.90, M2 17.12, M4 18.50 and M1 55.69; the petab_select 0.4.0 cross-check agrees. Honest reading: about 3.5 of the 6.95 Δ(-log L) over M3 comes from the 18 cells at 0 h and 16 h, which every model fits identically and which add 0.193 each through the shared sigma. On the 1-h and 2-h cells alone, ΔAIC to M3 is about 5. The set stays {M5}, but the weight is nearer 0.9. The set is relative to five hand-built candidates with no lattice; heterogeneity, M5 with processivity, and flexible count profiles were not tried.

**the observation model is stated explicitly, including how predicted amounts are mapped onto relative MS intensities at each time point** — `pass`

> The model is y_n(t_j) = sc_ctd_ms_tj \* \[CTD_n](t_j) + N(0, (sigma_ctd_ms \* sc_ctd_ms_tj)^2), on a linear scale, matching the lab note's description of noise that scales with each spectrum. The scales are t0 0.009986, t60 0.03971, t120 0.03844 and t960 0.008519 per µM, with sigma_ctd_ms 1.012. Residuals confirm the mapping (0.1027 = 2.555 sigma at 1 h), and substrate conservation drift is 1.4e-13. Not tested: equal MS response for every phospho-state.

**any mechanistic question the data cannot resolve is named as unresolved rather than decided** — `partial`

> computed_cannot_distinguish is empty and the FIM has no flat direction, so the round itself names nothing as unresolved. The open questions are structural, not parametric: positional arrangement is unobserved; nb moves from 1.812 (M3) to 0.314 (M5); no count-only candidate has a comparably flexible step-rate profile; there is no heterogeneous-substrate candidate; processivity was never combined with coupling; and equal MS response is assumed. These are named in cannot_distinguish here. Accepted as is, however, the selection would present the neighbour term as established.

**intent test: neighbour effect vs count cooperativity vs no dependence, and which the data can separate** — `partial`

> 'Nothing at all' is rejected for a homogeneous substrate (M1 ΔAIC 55.69, driven by the 1-h spectrum). Both terms are needed given each other (Δ(-log L) 9.56 and 6.95 for one parameter each). Neighbour-only edges count-only by ΔAIC 5.2, about half of which is sigma bookkeeping. The data do not establish that the second term is specifically an immediate-neighbour effect rather than a more flexible count dependence or substrate heterogeneity.

**intent test: the by-eye conclusion that P-TEFb is distributive rather than processive** — `partial`

> Processivity alone (M4, q_proc 0.2124) loses to the distributive winner by ΔAIC 18.50 (weight 0.0001), which is consistent with the by-eye conclusion. That loss is paid mainly at the end of the reaction: phos7 at 16 h (3.21) and phos8 at 2 h (1.62). M4 is the best candidate on the 1-h phos4-phos6 peaks. q_proc was held at 0 in M5 rather than estimated, so a processive component alongside coupling remains untested.
::::

::::{dropdown} What would fit better: the agent's ranked hypotheses for the remaining misfit
> 1 (highest). The per-step rate profile is still wrong at the top of the distribution. M5 is too slow into the upper counts at 1 h: phos5 +2.6 sigma (0.352 vs 0.250) and phos6 +2.0 sigma (0.158 vs 0.076). It is too fast into them at 2 h: phos8 -2.3 sigma (measured 0 vs 0.088) and phos7 -2.0 sigma (0.195 vs 0.273). Every coupled or processive candidate predicts 0.088-0.145 for phos8 at 2 h; only M1 (0.021) does not. Yet completion by 16 h needs a final step faster than M4's base rate of 0.0052/min. This points at a step rate that rises through mid-counts and drops for the last one or two repeats. That could be a non-monotone count dependence, or one intrinsically slow repeat such as a construct-terminal one. cp-type acceleration plus nb only approximates it. A count-only candidate with a separate final-step factor would test it.

> 2. The distribution broadens from the slow side, not by accelerating leaders. The 2-h low tail is heavier than every candidate gives: phos1 at 2 h is 0.191 measured vs 0.076-0.164 predicted, phos0 0.061 vs at most 0.051, and phos2 is +1.5 sigma under M5. Meanwhile the top end is empty. A slower substrate sub-population with independent sites broadens the distribution without populating phos7-phos8. That would be an alternative to cooperativity and would reopen 'the rate depends on nothing at all'. It needs two parameters (fraction and rate ratio) and is not in the candidate set.

> 3. A processive component on top of the coupling terms. M4 is the best candidate on phos4/phos5/phos6 at 1 h (rmse 0.021/0.042/0.021 vs M5's 0.048/0.103/0.082). It fails mainly at the end: 0.145 for phos8 at 2 h, and 3% left at phos7 at 16 h. M5 with q_proc free was never fitted.

> 4 (lowest). Observation layer. Equal MS response for every phospho-state is assumed and cannot be tested here. A monotone trend in response with phosphate number cannot both raise phos5-6 at 1 h and lower phos7-8 at 2 h, so it does not explain the main pattern. The 19 exact zeros behave as non-detections rather than Gaussian measurements. The phos8 zero at 2 h is not a detection artefact, though: phos0 at 2 h (0.061) was reported, which is below phos8's predicted 0.088. The zeros' main effect is on the size of the AIC margins, not on which model is selected.
::::

The verdict is a scientific claim, so a person signs it. The engine's own
default was to end the run here, since no misfit indicator was open; the
interpreter disagreed, and the ledger records that disagreement. The standing
answer settled it: one expansion round when the interpreter recommends revising.

:::{admonition} A gate · verdict (round 0)
:class: important
**The card asked:** Sign off this discrimination verdict? (it is a scientific claim)

**The answer:** **yes**, applied by the front end from the owner's standing answers.

**The note:**

> expand: yes
> follows the owner's standing answers of 2026-10-04 (evaluation_verdict: expand once when the interpreter recommends revise)
:::

## 4 · Round 1: what the model space lacked

### 4.1 · The expansion agent's diagnosis

The space-expansion agent ([the full prompt](prompts/revision.md)) reads where
**M5_neighbour_and_count** still misses, and what no candidate ever varied, and proposes new
**axes**: mechanisms to add to the winner and test. Its first proposal named two
axes that failed the pre-compile simulation check — every model carrying them
drove a state negative within the 16 hours —

> axis 'processive_adjacent_double_hit': every combination carrying 'processive_adjacent_double_hit' fails the pre-compile ladder: state goes negative over the horizon
> axis 'per_molecule_docking_route': every combination carrying 'per_molecule_docking_route' fails the pre-compile ladder: state goes negative over the horizon

so it was sent back once, automatically, and its second proposal passed:

| axis | priority | the question it tests |
|---|---|---|
| **product_load_inhibition** | 1 | Does the kinase slow as phosphate product accumulates? That would make the second hour slower relative to the first than a constant-activity kinase allows, on top of M5's count and neighbour coupling. |
| **processive_with_coupling** | 2 | Once count and neighbour coupling are present, does a processive component (several repeats modified per kinase encounter) still improve the fit, or does the distributive call hold? |


::::{dropdown} The agent's diagnosis, in full
:open:
> WHERE M5 STILL MISSES. Each peak at each time is its own n=1 series, so residual_forms carries no time shape. The structure has to be read across counts within each spectrum. All of M5's residuals above 1.5σ sit at 1 h and 2 h, and they form one timing distortion. At 60 min the data are further along than M5: obs_phos0 is −1.6σ (fewer unmodified molecules), obs_phos5 +2.6σ and obs_phos6 +2.0σ, and the spectrum's mean count is 2.48 measured against 2.33 simulated. At 120 min the data are behind: obs_phos7 is −2.0σ, obs_phos8 −2.3σ (0 measured, 0.088 simulated), obs_phos2 +1.5σ and obs_phos1 +0.7σ, and the mean count is 4.13 measured against 4.32. Each spectrum has its own free gain, so only shape is seen. Along M5's trajectory, shape is set by how far the reaction has run, and a constant-activity kinase fixes the 2-h/1-h ratio of that progress at 2; the data want less. Read through the mean count, the data look first-order between 1 h and 2 h: the 1-h rate (0.0062/min per site) predicts 4.20 at 2 h against 4.13 measured. M5 gets its 1-h breadth from count acceleration (cp 1.59), and that acceleration carries the 2-h spectrum past the data. The data therefore ask for broadening without mean acceleration. That is the signature of a brake that grows as the reaction proceeds, acting together with the coupling.
>
> WHAT WAS NEVER VARIED. Every candidate varied the per-step rate law. None varied the kinase: it was always constant and uninhibited, with mass-action kinetics. The interpretation set global effects aside because they cannot by themselves make a non-binomial shape. That is true. But on top of M5's coupling, a global brake changes the 1-h versus 2-h relation, which is exactly where M5 misses. The lab note states the kinase is not consumed and its amount is constant. So the brake is proposed as product occupancy: phosphate load, equal to ADP produced, sequesters free P-TEFb reversibly while total kinase stays conserved. Two variants are not proposed: kinase activity loss (the lab note rules it out) and substrate saturation of the kinase (it accelerates the late reaction, the opposite of what the residuals ask). This axis also bears on the intent. If cp moves toward 1 once the brake is on, part of the count-cooperativity evidence was compensating for timing.
>
> INTERPRETER'S HYPOTHESES.
> - #3 (processivity on top of coupling) is taken up as axis 2. The intent requires the distributive call to be tested, and so far it holds only against processivity alone (M4).
> - #1 is taken up through its timing part. The per-step version is left for construction: a count-only candidate with a flexible step-rate profile (rise, then brake for the last step), which was the interpreter's single move. It rewrites the J/Q rate laws, whose text is not in this payload. An axis would have to guess the current lines and would not nest. Until that candidate is fitted, an immediate-neighbour effect is not established.
> - #2 (static substrate heterogeneity) is also left for construction. It needs a second substrate chain plus observables that sum both chains, and one axis cannot carry an antimony patch and an assembly patch together. The complete 16-h spectrum (obs_phos7 at 960 min within 0.12σ under M5) also requires any slow subpopulation to finish within 16 h. That caps how much it can broaden the 1-h spectrum.
> - #4 (observation layer) is not testable here and is listed under beyond_this_data. The 19 exact zeros behave as non-detections and inflate the AIC margins, but no assembly patch expresses censoring.
>
> OTHER CHECKS.
> - No parameter is at or near a bound, and K_ATP is fixed at 50 µM. No axis releases a pinned parameter.
> - No replicate floor exists, so absolute adequacy cannot be judged.
> - Both axes of attempt 1 moved substrate between states outside the selected model's own transitions and drove states negative. They are dropped, not repaired. Neither new axis moves substrate mass: axis 1 moves only kinase between its free and product-occupied forms, and axis 2 frees the model's own Jp jump strength.
::::

::::{dropdown} What the agent says these data cannot separate, whatever is added
**An immediate-neighbour effect as such (modifying repeat i changes the rate at i±1), as distinct from a count-dependent step rate whose profile rises through the mid-counts and brakes near the end**

> MS resolves only the count, so the neighbour term reaches the data only through the count distribution. Its estimate crosses neutral between models (nb 1.81 in M3, 0.31 in M5), which suggests it is acting as a shape term for the step-rate profile. With four time points, no replicates and nine peaks per spectrum, a count-only profile with as many parameters can absorb the same imprint. A count-only flexible-profile candidate should still be fitted at construction, but even a tie would leave the question open.

**A phosphorylation-count-dependent MS ionisation and detection response**

> Each spectrum has one free gain, and the 0-h and 16-h spectra each contain a single species. Per-count response factors are therefore confounded with cp, nb and the kinetics. A single factor on phos8 alone would simply absorb the one informative cell (phos8 at 2 h).

**Loss of kinase activity during the incubation, rather than product inhibition, as the cause of the slower second hour**

> It would be separated from M5 only by the same timing signal the product-inhibition axis uses. With one condition sampled at 0, 1, 2 and 16 h, a time-dependent loss and a load-dependent brake both just slow the second hour relative to the first. The lab note states the kinase is unchanged, so it is not proposed as a competing axis.

**Inhibition by phosphorylated CTD, as opposed to ADP, as the product that occupies the kinase**

> In this assay each phosphate added produces one ADP, so both inhibitors equal Σ n·CTD_n at every time point.

**Genuine processivity (several repeats per binding event), as opposed to distributive release followed by rapid rebinding to the same molecule**

> Both produce clustered modification events. A count-level jump term cannot tell them apart at a single kinase and substrate concentration with no chase.
::::

Buying the round is another computation, so it is a gate. The base for the new
round is the criterion winner, M5_neighbour_and_count:

:::{admonition} A gate · buy an expansion round (round 0)
:class: important
**The card asked:** Buy this expansion and run another selection round?

**The answer:** **yes**, applied by the front end from the owner's standing answers.

**The note:**

> follows the owner's standing answers of 2026-10-04 (revision: y when the diagnosis names a missing mechanism; one expansion round)
:::

### 4.2 · Round 1: building and fitting the extension

Round 1 needs no new construction: the engine writes each bought axis onto the
base and keeps the base's own model, data reading and observation model. Only one
of the two axes became a candidate. `processive_with_coupling` would estimate the
processivity switch `q_proc` on the base, and the engine treated it as already
carried by the base, which holds `q_proc` at 0 — so processivity on top of the
coupling was **not** tested in this round. The note recorded with the compile
answer says so:

> follows the owner's standing answers of 2026-10-04 (assembly_cost_boundary: y). For the record: processive_with_coupling was bought at the expansion gate and named again at the scope card, but the engine left it out as already carried by the base; the base (M5) holds q_proc fixed at 0, and the axis only estimates it, so processivity on top of coupling is not tested in this round.


:::{admonition} A gate · model scope (round 1)
:class: important
**The card asked:** Is this the right model scope to spend compute on?

**The answer:** **yes**, applied by the front end from the owner's standing answers.

**The note:**

> tier: all
> follows the owner's standing answers of 2026-10-04 (scope with a positional base: y; the axes bought at the expansion gate, unchanged)
:::

:::{admonition} A gate · estimation budget (round 1)
:class: important
**The card asked:** Run the full estimation with 100 starts (projected ~205s wall)?

**The answer:** **yes**, applied by the front end from the owner's standing answers.

**The note:**

> follows the owner's standing answers of 2026-10-04 (budget: y, default starts, forward sensitivities)
:::

### 4.3 · S6 — fitting the base and the extension

Each candidate was fitted from 100 random starts. The **waterfall** shows the
final objective of every start, sorted: a flat plateau at the left means many
starts found the same best fit, so the optimum is trustworthy.

![Multistart waterfalls, round 1](figures/waterfalls_round1.svg)

| candidate | best −log L | `kmod` | `cp` | `nb` | `q_proc` | `kon_I` | `sigma_ctd_ms` |
|---|---|---|---|---|---|---|---|
| **base** | -91.20 | 0.00458 | 1.59 | 0.314 | 0 |  | 1.01 |
| **product_load_inhibition** | -95.40 | 0.00514 | 1.63 | 0.311 | 0 | 0.000909 | 0.89 |

At a bound in some fit: nothing.


### 4.4 · S7 — comparing again

![Selection, round 1](figures/selection_round1.svg)

**product_load_inhibition** wins with weight 0.961, ΔAIC 6.40 ahead of the base (the
round-0 winner). The interpretation agent of round 1
([the full prompt](prompts/interpretation_round_1.md)) again recommended
**`revise`**; the standing answer allowed one expansion round,
already used, so the run ended here, with the second recommendation recorded and
not bought.


::::{dropdown} The agent's diagnosis, in full
> SELECTED MECHANISM. product_load_inhibition is the round-0 winner M5_neighbour_and_count plus one global term, and it has the following parts. (a) It is distributive by construction: q_proc is fixed at 0.0. (b) The per-repeat rate is modulated by a neighbour factor nb = 0.311 and a count factor cp = 1.627. (c) ATP is saturating: K_ATP is fixed at 50 µM, while ATP never falls below about 2.2 mM (3 mM minus at most 0.8 mM consumed). (d) Free P-TEFb is inhibited by the accumulated phosphate load Σn·CTD_n, which equals the ADP produced. K_I = koff_I/kon_I = 1.0/0.000909 ≈ 1.1 mM, with koff_I fixed. If the rate tracks free P-TEFb, that is about 27% slowing at the ≈410 µM load reached by 2 h and about 42% at the 800 µM end point. Scores: AIC −172.80 and Akaike weight 0.961. The base is at ΔAIC 6.40 with weight 0.039. The indistinguishable set is {product_load_inhibition}. Only 2 of the 4 models in the declared lattice were visited, and the unvisited corners are untried, not rejected. There is no held-out series, so the selection rests on AIC alone.
>
> WHY IT WINS (attribution). ΔnLLH is 4.20, worth 8.40 likelihood units against 2 parameter units. Of that, 2.44 is earned in the 60- and 120-min spectra, as a net of gains and losses spread over nine peaks. Gains: phos5@60 +2.02 (winner 0.293, base 0.250, measured 0.352), phos2@120 +1.09, phos6@60 +0.87, phos4@60 +0.80, phos2@60 +0.64. Losses: phos1@60 −1.43, phos8@120 −1.21, phos6@120 −0.83, phos7@120 −0.64. The other 1.76 units come from the t0 and t960 points. Both models fit those to within 0.1σ, except phos7@960, which costs the winner 0.34. They pay only because the shared sigma fell from 1.012 to 0.890 µM. Switching kon_I off without refitting costs 188.8 AIC units. Refitted (that is, the base), it costs 6.4, because kmod drops from 0.00514 to 0.00458 to compensate. So the term mainly reshapes the progress curve rather than adding a large independent effect. The mean number of marks per molecule is 2.48 at 1 h and 4.13 at 2 h. First-order depletion of repeats at a constant ≈0.0062 min⁻¹ already predicts 4.19 at 2 h, so the overall deceleration does not by itself demand a global brake.
>
> OBSERVATION MODEL. Each peak's intensity is sc_t × the amount in µM, with a linear transformation. The scales are sc_ctd_ms_t0 = 0.00999, sc_ctd_ms_t60 = 0.0404, sc_ctd_ms_t120 = 0.0393 and sc_ctd_ms_t960 = 0.0086 per µM. For 100 µM of conserved substrate these give totals of 1.00, 4.04, 3.93 and 0.86, against measured 1.00, 4.07, 3.79 and 0.85. Noise is additive normal with one sigma_ctd_ms = 0.890 µM for all peaks, scaled by the same sc_t. In intensity units that is σ = 0.0089, 0.036, 0.035 and 0.0077 at the four time points.
>
> FIT QUALITY. Sigma is fitted and there are no replicates, so χ² = 36 = n_data by construction and no absolute goodness-of-fit test exists. About 35 of the 36 χ² units sit in the 18 points at 60 and 120 min, because each t0 and t960 spectrum is fitted trivially by its own scale factor. The fitted σ is therefore about √2 smaller than the scatter of the informative points. That inflates every σ-unit below and sharpens the likelihood behind ΔAIC. On the 60/120-min points alone the margin would be roughly 2×2.44−2 ≈ 2.9 AIC units, weight about 0.8: still above the 2.0 threshold, so the selection stands, but less sharply than 0.961 suggests.
>
> WHAT IS STILL WRONG. All residuals are single unreplicated points, so the case rests on sign agreement across the two informative spectra.
> - High counts: at both 60 and 120 min the winner puts mass at 7 and 8 marks that the data lack. Residuals: phos7@60 −0.66σ, phos8@60 −0.10σ, phos7@120 −2.35σ, phos8@120 −2.78σ (the last two are the largest in the fit).
> - Middle counts: it puts too little at 5–6 marks: phos5@60 +1.65σ, phos6@60 +1.64σ, phos6@120 +2.05σ.
> - Late time: it leaves about 0.9% of molecules at 7 marks at 16 h (phos7@960 −0.97σ), and the data have none.
> - The zero at phos8@120 is not a detection floor. Measured 8-mark CTD is 0 at 2 h while 7-mark CTD is 0.19, and a 0-mark peak of 0.061 was detected in the same spectrum.
> - So the data want the last one or two modifications specifically slowed early, yet complete by 16 h. The global product-load term brakes every step instead: it raised phos8@120 from 0.088 to 0.097.
> - A weaker, isolated residual at 60 min: phos1 +2.18σ, with phos0 −1.57σ and phos3 −1.48σ.
>
> MECHANISM ANSWERS. Round-0 log-likelihoods are given. AICs below assume each variant adds one parameter to a 6-parameter independent model, consistent with the 8-parameter M5. ΔAIC to the winner: M5 6.4, neighbour-only M3 18.3, count-only M2 23.5, processive-only M4 24.9, independent M1 62.1.
> 1. 'Rate depends on nothing' is rejected (−log L −61.36 vs −95.40). The measured distributions are broader than binomial: the variance of marks per molecule is 2.57 against 1.71 at 1 h, and 2.63 against 2.00 at 2 h.
> 2. Neighbour-only beats count-only by about 5.2 AIC, and both together beat neighbour-only by about 11.9. Dropping cp costs 6.95 nllh units and dropping nb 9.56. nb and cp move by less than 3% between the two round-1 candidates.
> 3. But intact-mass MS sees only the configuration-averaged rate at each count. A neighbour effect projects onto an effective count-dependent rate profile, and M5's need for two shape parameters is what a two-parameter or free per-count count-only profile could also supply. That model was not fitted, so 'neighbour over count' holds only against the one-parameter count form.
> 4. Distributive vs processive was not tested on the selected structure, because q_proc is fixed at 0.0 in both round-1 candidates. The only processive model fitted, M4 (processive with otherwise independent rates), loses to M5 by about 18.5 AIC, but it changes two things at once. Against count-only M2 it differs by only about 1.4 AIC, below the 2.0 threshold. Processivity and positive count cooperativity both broaden count distributions, and the data do not separate them as single additions. The distributions are unimodal at 60 and 120 min, with no unmodified-plus-heavily-modified bimodality. That is consistent with the by-eye distributive reading but does not exclude a partial processive component.
> 5. The cause of the global slowdown is not identifiable. ADP and phospho-CTD both equal Σn·CTD_n by stoichiometry, and in a single condition progress is monotone in time, so time-dependent kinase inactivation is confounded with both.
>
> IDENTIFIABILITY. The local FIM is full rank with a 3.72-decade eigenvalue spread (threshold 8) and no flat directions. The smallest eigenvector is dominated by kon_I (0.993, with kmod at 0.111). Profiles were not computed, so no parameter interval is known. kon_I is the least-determined parameter: its K_I lies above the maximal load of 0.8 mM, so only the initial slope of inhibition is seen.
>
> RECOMMENDATION. Revise, with one move: free q_proc on product_load_inhibition. This is the test of the distributive conclusion that the intent asks for and that the winner currently assumes. The next move after that is a count-only model with a high-occupancy slowdown term, fitted against nb. It targets the 7–8-mark misfit and tests whether the neighbour term is separable from count dependence at all.
::::

::::{dropdown} Its answer, criterion by criterion
**the selected mechanism is reported with its information criterion, its Akaike weight and the indistinguishable set, honestly, whatever they turn out to be** — `pass`

> product_load_inhibition: AIC −172.80, Akaike weight 0.961. Base (M5_neighbour_and_count) at ΔAIC 6.40, weight 0.039. Indistinguishable set {product_load_inhibition}, decisive against the 2.0 / 0.7 thresholds. Stated with its caveats: only 2 of 4 lattice models were visited; there is no held-out data; 1.76 of the 4.20 nllh units are collected at t0/t960 only through the shared sigma, and the 60/120-min spectra alone would give roughly 2.9 AIC units.

**the observation model is stated explicitly, including how predicted amounts are mapped onto relative MS intensities at each time point** — `pass`

> Intensity = sc_t × amount (µM), linear, with one free scale per time point: 0.00999, 0.0404, 0.0393 and 0.0086 at 0, 60, 120 and 960 min. Model totals are 1.00, 4.04, 3.93 and 0.86 against measured 1.00, 4.07, 3.79 and 0.85. Noise is additive normal with one sigma_ctd_ms = 0.890 µM for every peak, scaled by the same sc_t, giving σ = 0.0089, 0.036, 0.035 and 0.0077 intensity units.

**any mechanistic question the data cannot resolve is named as unresolved rather than decided** — `partial`

> Named here as unresolved: neighbour versus flexible count dependence; processive versus distributive on the winning structure; count cooperativity versus processivity versus heterogeneity (M2 vs M4 ≈1.4 AIC); ADP versus phospho-CTD versus kinase inactivation; kon_I versus koff_I; K_ATP; positional pattern. But the run itself decided two of these by construction (q_proc fixed at 0.0, koff_I fixed at 1.0). computed_cannot_distinguish is empty, and no profiles were computed. Processivity is therefore untested, not shown to be undecidable.

**test whether modification of a repeat changes its neighbours' rate, against count cooperativity or nothing at all** — `partial`

> 'Nothing at all' is rejected by about 62 AIC units. Neighbour-only beats count-only by about 5.2 AIC, and neighbour+count beats neighbour-only by about 11.9 (one parameter per term assumed). This separates neighbour from count only against the one-parameter count form; a count-only model with two or more shape parameters, the alternative counts-only data cannot a priori exclude, was not fitted.

**test the earlier by-eye conclusion that P-TEFb acts distributively rather than processively** — `partial`

> The distributions are unimodal at 60 and 120 min, consistent with distributive action. However, q_proc is fixed at 0.0 in the selected model. The only processive model (M4, −log L −80.95) loses to M5 by about 18.5 AIC while also lacking the neighbour and count terms, and it ties count-only M2 within about 1.4 AIC. Processivity on top of the selected structure has not been fitted.
::::

::::{dropdown} What would fit better: the agent's ranked hypotheses for the remaining misfit
> 1 (best supported, unaddressed by any visited model). A slowdown specific to the last one or two modifications is missing: the per-repeat rate falls once a molecule carries about 6 or more marks, or terminal/sequence-divergent repeats are poor substrates. Evidence: n=7,8 are over-predicted at both informative times (phos7@120 −2.35σ, phos8@120 −2.78σ, phos7@60 −0.66σ) and n=5–6 under-predicted (phos5@60 +1.65σ, phos6@60 +1.64σ, phos6@120 +2.05σ). Measured 8-mark CTD is 0 at 2 h while a 0.061 peak was detected in the same spectrum, and conversion is complete by 16 h. The global product-load term does not supply this: it worsened phos8@120 (0.088 → 0.097) and created phos7@960 −0.97σ. Count-dependent and position-dependent versions cannot be separated from intact-mass counts; middle-down or positional MS, or single-repeat mutants, would separate them.

> 2. The product-load term partly stands in for distribution shape rather than reporting an independent inhibitory process. Evidence: the mean marks per molecule (2.48 at 1 h, 4.13 at 2 h) matches constant-rate first-order site depletion (≈0.0062 min⁻¹ predicts 4.19 at 2 h). Switching kon_I off unrefitted costs 188.8 AIC units, but refitted it costs only 6.4, with kmod compensating from 0.00514 to 0.00458. Of the 4.20 nllh gain, only 2.44 is earned at 60/120 min, spread as wins and losses over nine peaks. ADP inhibition, phospho-CTD inhibition and time-dependent kinase inactivation would all appear this way in one condition. An ADP spike-in, a second ATP/ADP ratio, or late enzyme addition would separate them.

> 3. A partial processive component, or molecule-to-molecule rate heterogeneity (not in the model space), contributes to the broader-than-binomial distributions now carried by cp alone. Evidence: variance of marks 2.57 vs 1.71 binomial at 1 h and 2.63 vs 2.00 at 2 h. Processive-only and count-only additions to the independent model differ by only ≈1.4 AIC (−log L −80.95 vs −81.64). Untested on the selected structure because q_proc is fixed at 0.0; freeing it there is the separating move.

> 4 (lowest as a cause of mechanism misfit). Observation layer: absent peaks are recorded as exact zeros (phos7@60, phos8@60, phos8@120 and every absent peak at t0/t960), which additive Gaussian noise cannot produce. The 18 trivially fitted t0/t960 points also shrink the shared sigma by about √2 relative to the informative scatter, inflating every σ-unit (phos8@120 would be ≈−2.0σ) and the likelihood margin. The noise model is the study team's stated decision and is not changed here. The phos8@120 zero is not a detection-floor artefact, because a smaller peak (phos0@120 = 0.061) was detected in the same spectrum.
::::

:::{admonition} A gate · verdict (round 1)
:class: important
**The card asked:** Sign off this discrimination verdict? (it is a scientific claim)

**The answer:** **yes**, applied by the front end from the owner's standing answers.

**The note:**

> expand: no
> follows the owner's standing answers of 2026-10-04 (evaluation_verdict: one expansion round, already used in round 0; the interpreter's second revise is recorded, not bought)
:::

## 5 · The models side by side

Every model the run fitted, on the same data and the same likelihood, so the AIC
values compare across rounds (ΔAIC against the selected model; the base of round
1 is M5_neighbour_and_count, refitted, and appears once):

| model | round | mechanism terms (fitted) | parameters | −log L | ΔAIC |
|---|---|---|---|---|---|
| **M1_independent** | 0 | — (independent repeats) | 6 | -61.36 | 62.09 |
| **M2_count_coop** | 0 | `cp` = 1.14 | 7 | -81.64 | 23.52 |
| **M3_neighbour** | 0 | `nb` = 1.81 | 7 | -84.25 | 18.30 |
| **M4_processive** | 0 | `q_proc` = 0.212 | 7 | -80.95 | 24.90 |
| **M5_neighbour_and_count** | 0 | `cp` = 1.59, `nb` = 0.314 | 8 | -91.20 | 6.40 |
| **product_load_inhibition** (selected) | 1 | `cp` = 1.63, `nb` = 0.311, `kon_I` = 0.000909 | 9 | -95.40 | 0.00 |

How each fits the data — bars: measured; lines: each candidate's prediction at its
best fit, the selected model of the round drawn thicker. Round 0:

![Fits of every candidate, round 0](figures/fits_round0.svg)

Independent repeats (grey) cannot produce the broad spectrum at 60 minutes, where
unmodified and heavily modified molecules coexist; the coupled models can. Round 1
— the base against the extension:

![Fits of the base and the extension, round 1](figures/fits_round1.svg)

The extension changes the shape only slightly: it lets the kinase slow as product
accumulates, which moves the 60- and 120-minute spectra closer to the data, as the
interpretation's attribution details.


## 6 · The answer, and what stays open

**Selected:** product_load_inhibition — M5_neighbour_and_count + product_load_inhibition — by AIC over
every candidate fitted, decisively (Akaike weight 0.961), with no other model
indistinguishable. Rates that depend on nothing are
rejected; both coupling terms are away from neutral; and a slowdown that grows as
product accumulates improves the fit, though what causes it is listed below as
undetermined. What the coupling *is* — a neighbour effect, or a count-dependent
rate with a more flexible profile — these count-only data cannot decide, and the
run says so rather than claiming it.

**Against the success criteria the user set at the start** (round 1):

- `pass` — the selected mechanism is reported with its information criterion, its Akaike weight and the indistinguishable set, honestly, whatever they turn out to be
- `pass` — the observation model is stated explicitly, including how predicted amounts are mapped onto relative MS intensities at each time point
- `partial` — any mechanistic question the data cannot resolve is named as unresolved rather than decided
- `partial` — test whether modification of a repeat changes its neighbours' rate, against count cooperativity or nothing at all
- `partial` — test the earlier by-eye conclusion that P-TEFb acts distributively rather than processively

**Determined nonetheless:**

- the mechanism of product_load_inhibition
- Rates that depend on nothing are rejected. The independent distributive model (round 0, −log L −61.36) is 34 log-likelihood units, about 62 AIC units at 6 parameters, below the selected model. The measured count distributions are broader than binomial: variance of marks 2.57 vs 1.71 at 1 h and 2.63 vs 2.00 at 2 h.
- Within the neighbour+count parametrisation, both terms are away from neutral. Removing cp costs 6.95 and removing nb 9.56 log-likelihood units (round 0). Their MLEs move by less than 3% when product inhibition is added (nb 0.314 → 0.311, cp 1.588 → 1.627).
- The per-time-point intensity scales are pinned by the spectrum totals for 100 µM of conserved substrate: sc_ctd_ms_t0 = 0.00999, sc_ctd_ms_t60 = 0.0404, sc_ctd_ms_t120 = 0.0393, sc_ctd_ms_t960 = 0.0086 per µM. They reproduce the measured totals 1.00, 4.07, 3.79 and 0.85 to within 4%.
- Fully modified CTD is absent at 2 h while 7-mark CTD is about 5% of molecules and a 0.061 peak is detected in the same spectrum; conversion is complete by 16 h, with 8-mark CTD the only species at 960 min.

**Declared undetermined** — stated because they are true, not because the run failed:

- Neighbour-dependent rates (nb) versus a count-dependent rate profile with more than one shape parameter. Counts-only MS sees the configuration-averaged rate per count. The round-0 preference for neighbour-only over count-only (≈5.2 AIC) and for neighbour+count over either compares one-parameter encodings only; a flexible count-only profile was never fitted.
- Processive versus distributive modification on the selected structure. q_proc is fixed at 0.0 in both round-1 candidates. The only processive model fitted (M4_processive in round 0, −log L −80.95) lacks the neighbour and count terms, so its ≈18.5 AIC deficit to M5 conflates processivity with rate modulation.
- Positive count cooperativity versus processivity (and versus molecule-to-molecule rate heterogeneity, not in the model space) as the source of the broader-than-binomial count distributions. Count-only and processive-only additions to the independent model differ by ≈1.4 AIC units (−log L −81.64 vs −80.95), below the 2.0 threshold.
- The identity of the global slowdown. Inhibition of P-TEFb by ADP and by phosphorylated CTD both equal Σn·CTD_n by stoichiometry, and in a single condition both are confounded with time-dependent loss of kinase activity.
- kon_I and koff_I separately. koff_I is fixed at 1.0, so only K_I = koff_I/kon_I ≈ 1.1 mM is estimated. With samples 60 min or more apart the binding kinetics are not seen, and K_I above the maximal 0.8 mM load exposes only the initial slope of inhibition.
- K_ATP and any ATP-depletion effect. K_ATP is fixed at 50 µM, and ATP stays at or above about 2.2 mM throughout, so the data carry no information on ATP affinity.
- Which repeats carry the marks, including whether the deficit of 7- and 8-mark molecules at 60–120 min is position-dependent (terminal or divergent repeats being slower substrates) or count-dependent. Intact-mass MS does not resolve positions.


## 7 · Cost, time and provenance

| | |
|---|---|
| agent calls | 19 · $25.40 |
| answers at gates | 15 (5 at the card, 10 from standing answers) · policy `strict` (every gate asks a person) |
| redrafts | 3 automatic (the expansion agent twice, the expansion agent once), one at the person's request, and one stop and re-entry |
| wall time | 21.9 h in 17 sessions · agents 2.8 h · computation 2.0 h · waiting at gates 8.7 h |
| code | commit `0f8649d` (with uncommitted changes) |
| versions | python 3.13.12, pypesto 0.6.0, amici 1.0.1, petab 0.8.2, petab_select 0.4.0, numpy 2.5.2, scipy 1.18.1 |

## 8 · Where to go next

- **[Explore the record yourself](explore_record.ipynb)** — open this run's file in
  Python and find every item on this page.
- **[All prompts](prompts/index.md)** — every agent's full prompt and reply.
- **The engine's own reports for this run** —
  <a href="../../ctd_blind_reports/ANALYSIS.html">ANALYSIS</a> ·
  round 0: <a href="../../ctd_blind_reports/round_0/report/model_report.html">model report</a>,
  <a href="../../ctd_blind_reports/round_0/report/report.html">evaluation report</a> ·
  round 1: <a href="../../ctd_blind_reports/round_1/report/report.html">evaluation report</a>
